# FedKG-Maint | Unified, Auditable Pipeline (Phase 1 → Phase 4)

**Single Google Colab notebook — Azure Predictive Maintenance / 5 simulated factories.**

This notebook consolidates and improves the provided Data Preparation v3, Centralized Baselines v3, Phase 3 Privacy/Utility LocalNorm, Phase 3 Closure, and Local Knowledge Graphs v1. It does not invent outcomes: all scores, audits, graphs and reports are generated only when run against real CSVs. Prior notebook outputs are not reused as acceptance evidence.

**Execution order**
1. Dataset fingerprint + temporal labels + factory manifest + feature engineering + leakage tests.
2. Validation-only centralized and local-only baselines; artifact selection lock.
3. Validation-only FedAvg, FedProx, ClippedFedAvg, DP-FedAvg + matched compact central reference.
4. ONE held-out-site evaluation, paired machine bootstrap, privacy accounting and communication estimates.
5. Five local RDF graphs + shared ontology + strict as-of graph retrieval tests.
6. Consolidated audit, limits and ready-to-report results.

**Security and interpretation:** Simulation uses centralized process memory; local normalization and client-update noise are *not* physically distributed data confidentiality. The DP ε applies only to the client-update Gaussian release under its declared adjacency/participation assumptions, not to preprocessing, validation, graph release, or the whole system. Factory 5 is unseen to supervised training but supplies *unlabeled earlier-window calibration* to its scaler. Site evaluation is not zero-shot. The Azure dataset is the only data source; the NASA C-MAPSS portion of the original assignment is not performed here.

**No raw-data deletion:** Writes only inside a named `unified_runs/<RUN_ID>/` folder. Existing prep_v2/prep_v3 and earlier experiments are not deleted or silently reused.

### تشغيل مختصر

1. افتح الملف في **Google Colab**، وتأكد أن ملفات Azure الخمسة موجودة داخل `MyDrive/FedKG-Maint/data/raw/`؛ وإلا سيحاول تحميلها من Kaggle.
2. غيّر `DRIVE_PROJECT` عند الحاجة، وحدد `RUN_ID` جديدًا لكل تجربة نهائية مستقلة. شغّل **Runtime → Run all** بالترتيب.
3. لا تستخدم `FEDKG_FAST_SMOKE=1` لإخراج نتائج نهائية. نتائج الاختبار تُعرض بعد تثبيت إعدادات التدريب، ولا يتم ضبط المعلمات بناءً عليها.
4. بعد اكتمال التشغيل، راجع `FINAL_INTEGRATED_REPORT.md` و`FINAL_INTEGRATED_AUDIT.json` داخل `unified_runs/<RUN_ID>`، إلى جانب ملفات Predictions وGraphs وSelection Locks.

**تنبيه:** هذه Notebook كاملة من ناحية الكود، لكنها **لم تُشغَّل على بيانات Drive الخاصة بك من هذه الجلسة**. نجاح التشغيل والنتائج لا يمكن الجزم بهما دون تنفيذها على البيانات الفعلية. لا تحذف ملفاتك القديمة؛ الحزمة تكتب في مجلد تشغيل مستقل.

## 0. Environment, Drive folder, run configuration

Google Colab: **Runtime → Run all**, authorize your own Drive mount. The expected existing files are under `MyDrive/FedKG-Maint/data/raw/PdM_*.csv`; if missing, `kagglehub` downloads the public Azure source. Change `DRIVE_PROJECT` if your Drive uses another folder. Change `RUN_ID` when executing a genuinely new experiment. Do **not** pick values after inspecting test results.

In [ ]:
import os, sys, subprocess, importlib.util, random, json, datetime, hashlib, time, warnings
from pathlib import Path
for package, module in [('pyarrow','pyarrow'),('rdflib','rdflib'),('kagglehub','kagglehub'),('dp-accounting','dp_accounting')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
import numpy as np, pandas as pd, sklearn, joblib, torch, rdflib
from IPython.display import display
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_PROJECT = Path('/content/drive/MyDrive/FedKG-Maint')
except ImportError:
    DRIVE_PROJECT = Path(os.environ.get('FEDKG_DRIVE_ROOT', './FedKG-Maint')).expanduser().resolve()
RUN_ID = os.environ.get('FEDKG_RUN_ID', 'unified_v1_20261025')
assert RUN_ID.replace('_','').replace('-','').isalnum(), 'Use simple alphanumeric RUN_ID.'
PROJECT_DIR = DRIVE_PROJECT / 'unified_runs' / RUN_ID
RAW_DIR = DRIVE_PROJECT / 'data' / 'raw'
PROJECT_DIR.mkdir(parents=True, exist_ok=True); RAW_DIR.mkdir(parents=True, exist_ok=True)
if (PROJECT_DIR / 'RUN_COMPLETE.json').exists():
    raise RuntimeError(f'This run is COMPLETE: {PROJECT_DIR}. Choose a new RUN_ID to avoid overwriting a closed evaluation.')
SEEDS = [42, 43, 44]
# FAST_SMOKE is for software checks only; fast-mode results are NOT publication results.
FAST_SMOKE = os.environ.get('FEDKG_FAST_SMOKE','0') == '1'
BOOTSTRAP_B = 100 if FAST_SMOKE else 1000
ROUNDS = 2 if FAST_SMOKE else 3
LOCAL_EPOCHS = 1
CLIP_NORM = 0.10
NOISE_MULTIPLIER = 1.0
DP_DELTA = 1e-5
EPSILON_CAP = 10.0
MODEL_POLICY = 'local'  # authoritative privacy-compliant scaler policy
ENABLE_HGB = True
os.environ['FEDKG_ROOT'] = str(PROJECT_DIR)
print('Drive:', DRIVE_PROJECT)
print('Run root:', PROJECT_DIR)
print('Dataset:', RAW_DIR)
print('Mode:', 'SMOKE TEST -- NOT FINAL RESULTS' if FAST_SMOKE else 'FULL RUN')

# Phase 1 — Verified data preparation

Adapted from `FedKG_Maint_Data_Preparation_v3_Revised(2).ipynb` and kept intact where the existing leakage and label cross-checks provide stronger independent evidence. No audit CSVs from earlier versions are trusted as current.

In [ ]:
# 1.1  Setup: environment, configuration, directories, seeds, check registry
import os, sys, json, random, hashlib, shutil, subprocess, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

# PROJECT_DIR and RAW_DIR come from the unified bootstrap cell.
# ---- single project configuration ---------------------------------------------------------
SEED = 42
HORIZON_HOURS = 24
ROLLING_WINDOWS = {"24h": 24, "7d": 168}           # window name -> number of hourly rows
SENSORS = ["volt", "rotate", "pressure", "vibration"]
TARGET = "label_failure_24h"
N_FACTORIES, MACHINES_PER_FACTORY = 5, 20
HELD_OUT = "factory_5"
TRAINING_FACTORIES = [f"factory_{i}" for i in range(1, N_FACTORIES + 1) if f"factory_{i}" != HELD_OUT]
TRAIN_FRACTION = 0.80                              # share of the timeline used for training
VAL_FRACTION_OF_REMAINDER = 0.50                   # remainder is split validation / late test
FRAGILE_EVENT_THRESHOLD = 30                       # fewer distinct failure events => unstable metrics
HELD_OUT_CALIBRATION_FRACTION = 0.50                # earlier, unlabeled calibration window for local_factory_5

H = pd.Timedelta(hours=HORIZON_HOURS)
H64 = H.to_timedelta64()

# ---- write to a new versioned directory; old prep_v2 outputs remain untouched -------
RAW_DIR = DRIVE_PROJECT / "data" / "raw"
PREP_DIR = PROJECT_DIR / "prep_v3"
PROC_DIR, SCALER_DIR = PREP_DIR / "processed", PREP_DIR / "scalers"
CFG_DIR, REP_DIR, MAN_DIR = PREP_DIR / "configs", PREP_DIR / "reports", PREP_DIR / "manifests"
for d in [RAW_DIR, PROC_DIR, SCALER_DIR, CFG_DIR, REP_DIR, MAN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED)

# ---- check registry: every check is recorded and stops the run on failure -----------------
CHECKS = {}
def record(check_id, ok, detail=""):
    CHECKS[check_id] = {"status": "PASS" if ok else "FAIL", "detail": str(detail)}
    print(f"[{'PASS' if ok else 'FAIL'}] {check_id}: {detail}")
    if not ok:
        raise AssertionError(f"Check failed: {check_id} — {detail}")

print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__)
print("Project directory:", PROJECT_DIR)
print("Output directory :", PREP_DIR)

## 2. Data acquisition and immutable fingerprints

The five Azure PdM CSVs are saved to `FedKG-Maint/data/raw/`. If they are missing or have unexpected row counts, the dataset is downloaded. Current SHA-256 hashes are checked against an existing manifest (`prep_v3/` first, then an earlier `prep_v2/` manifest if available). Existing fingerprints are never silently overwritten.


In [ ]:
# 2.1  Acquire the five raw CSVs and verify their fingerprints
EXPECTED_ROWS = {
    "PdM_telemetry.csv": 876_100, "PdM_errors.csv": 3_919, "PdM_failures.csv": 761,
    "PdM_maint.csv": 3_286, "PdM_machines.csv": 100,
}


def raw_is_ready():
    for name, n in EXPECTED_ROWS.items():
        p = RAW_DIR / name
        if not p.exists():
            return False
        try:
            if len(pd.read_csv(p, usecols=[0])) != n:
                return False
        except Exception:
            return False
    return True


if not raw_is_ready():
    try:
        import kagglehub
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
        import kagglehub
    src_root = Path(kagglehub.dataset_download("arnabbiswas1/microsoft-azure-predictive-maintenance"))
    for name in EXPECTED_ROWS:
        found = list(src_root.rglob(name))
        assert len(found) == 1, f"{name}: expected exactly one match, found {len(found)}"
        shutil.copy2(found[0], RAW_DIR / name)
    print("Downloaded raw data to", RAW_DIR)
else:
    print("Raw data already present:", RAW_DIR)


def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()


checksums = {
    name: {"rows": EXPECTED_ROWS[name], "bytes": (RAW_DIR / name).stat().st_size,
           "sha256": sha256_of(RAW_DIR / name)}
    for name in EXPECTED_ROWS
}
record("DATA-1 raw files present with expected row counts", raw_is_ready(),
       ", ".join(f"{n.replace('PdM_', '').replace('.csv', '')}={v['rows']:,}" for n, v in checksums.items()))

# Prefer this version's immutable reference. If this is the first prep_v3 run, compare
# against a previous prep_v2 manifest when available. Do not overwrite a mismatching reference.
checksum_path = MAN_DIR / "raw_data_checksums.json"
reference_candidates = [
    checksum_path,
]
reference_path = next((p for p in reference_candidates if p.exists()), None)
if reference_path is not None:
    reference = json.loads(reference_path.read_text())
    mismatches = []
    for name, actual in checksums.items():
        expected = reference.get(name)
        if expected is None:
            mismatches.append(f"{name}: missing from reference manifest")
            continue
        for field in ("rows", "bytes", "sha256"):
            if expected.get(field) != actual[field]:
                mismatches.append(f"{name}: {field} differs")
    record("DATA-2 raw SHA-256 fingerprints match the saved reference",
           not mismatches,
           f"reference={reference_path}; " + ("all fingerprints match" if not mismatches else "; ".join(mismatches)))
    # When using a previous-version reference, copy it only after successful comparison.
    if reference_path != checksum_path:
        checksum_path.write_text(json.dumps(reference, indent=2))
else:
    # First-ever run: initialize a baseline. Future runs will compare against it.
    checksum_path.write_text(json.dumps(checksums, indent=2))
    record("DATA-2 initial raw fingerprint baseline created", True,
           "No earlier checksum manifest existed; subsequent runs will fail if any hash changes.")

print("Fingerprint manifest:", checksum_path)

## 3. Data-quality audit
Missing values, duplicates, regular hourly telemetry grid, alignment of failure events with the telemetry grid, and identifier consistency across the five tables.

In [ ]:
# 3.1  Load raw tables and run the data-quality audit
tele = pd.read_csv(RAW_DIR / "PdM_telemetry.csv", parse_dates=["datetime"])
errors = pd.read_csv(RAW_DIR / "PdM_errors.csv", parse_dates=["datetime"])
failures = pd.read_csv(RAW_DIR / "PdM_failures.csv", parse_dates=["datetime"])
maint = pd.read_csv(RAW_DIR / "PdM_maint.csv", parse_dates=["datetime"])
machines = pd.read_csv(RAW_DIR / "PdM_machines.csv")
for _t in (tele, errors, failures, maint):                      # one datetime resolution everywhere (needed by merge_asof)
    _t["datetime"] = _t["datetime"].astype("datetime64[ns]")
tables = {"telemetry": tele, "errors": errors, "failures": failures, "maintenance": maint, "machines": machines}

# --- DQ-1: missing values and exact duplicate rows
audit = pd.DataFrame({n: {"rows": len(t), "missing": int(t.isna().sum().sum()), "duplicate_rows": int(t.duplicated().sum())}
                      for n, t in tables.items()}).T
print(audit.to_string())
record("DQ-1 no missing values and no duplicate rows", int(audit["missing"].sum() + audit["duplicate_rows"].sum()) == 0)

# --- DQ-2: telemetry is a complete, regular hourly grid for every machine
tele = tele.sort_values(["machineID", "datetime"]).reset_index(drop=True)
per_machine = tele.groupby("machineID")["datetime"].agg(["size", "min", "max"])
steps = tele.groupby("machineID")["datetime"].diff().dropna()
record("DQ-2 hourly grid, identical coverage for all machines",
       per_machine["size"].nunique() == 1 and (steps == pd.Timedelta(hours=1)).all()
       and not tele.duplicated(["machineID", "datetime"]).any(),
       f"{len(per_machine)} machines x {int(per_machine['size'].iloc[0]):,} rows, "
       f"{tele['datetime'].min()} -> {tele['datetime'].max()}")

# --- DQ-3: every failure timestamp lies on the telemetry grid of its machine
chk = failures.merge(tele[["machineID", "datetime"]], on=["machineID", "datetime"], how="left", indicator=True)
record("DQ-3 failure events align with telemetry timestamps", bool((chk["_merge"] == "both").all()),
       f"{len(failures)} failure rows")

# --- DQ-4: identifiers are consistent across tables
ids = set(machines["machineID"])
record("DQ-4 machine identifiers consistent across tables",
       set(tele["machineID"]) == ids and set(errors["machineID"]) <= ids
       and set(failures["machineID"]) <= ids and set(maint["machineID"]) <= ids,
       f"{len(ids)} machines")

# --- descriptive summary
fail_events = (failures[["machineID", "datetime"]].drop_duplicates()
               .sort_values(["machineID", "datetime"]).reset_index(drop=True))
print("\nDate ranges:")
for n, t in [("telemetry", tele), ("errors", errors), ("failures", failures), ("maintenance", maint)]:
    print(f"  {n:12s} {t['datetime'].min()} -> {t['datetime'].max()}")
print(f"\nFailure rows: {len(failures)} | distinct (machine, timestamp) failure events: {len(fail_events)} "
      f"| machines with failures: {failures['machineID'].nunique()}")
print("Failures by component:", failures["failure"].value_counts().sort_index().to_dict())
print("Machine models:", machines["model"].value_counts().sort_index().to_dict())

## 4. Target construction and verification

`label_failure_24h = 1` iff the next failure of the same machine occurs strictly after `t` and no later than `t + 24h`. A failure recorded exactly at `t` is **not** a future failure for row `t`.

The label is computed by one code path (`numpy.searchsorted`) and then **re-computed by two independent paths** (`merge_asof` and a brute-force check on a random sample). All three must agree exactly. The last 24 rows of each machine are removed because their full 24 h future is not observable (treating them as negatives would invent information).

In [ ]:
# 4.1  Label construction (primary path) + two independent verifications + boundary handling
def labels_searchsorted(tele_sorted, failures_df):
    """Primary path: first failure strictly after t via numpy.searchsorted."""
    lab = np.zeros(len(tele_sorted), dtype=np.int8)
    times = tele_sorted["datetime"].to_numpy()
    fails = {m: np.sort(g["datetime"].to_numpy()) for m, g in failures_df.groupby("machineID")}
    for m, idx in tele_sorted.groupby("machineID").indices.items():
        ft = fails.get(m)
        if ft is None:
            continue
        t = times[idx]
        pos = np.searchsorted(ft, t, side="right")          # index of first failure strictly after t
        ok = pos < len(ft)
        lab[idx[ok]] = ((ft[pos[ok]] - t[ok]) <= H64).astype(np.int8)
    return lab

def labels_merge_asof(tele_sorted, failures_df):
    """Independent path: pandas merge_asof (forward, exact matches excluded)."""
    left = tele_sorted[["machineID", "datetime"]].copy()
    left["_row"] = np.arange(len(left))
    left = left.sort_values("datetime")
    right = failures_df[["machineID", "datetime"]].drop_duplicates().rename(columns={"datetime": "fail_time"})
    right["datetime"] = right["fail_time"]
    right = right.sort_values("datetime")
    m = pd.merge_asof(left, right, on="datetime", by="machineID", direction="forward", allow_exact_matches=False)
    m = m.sort_values("_row")
    return ((m["fail_time"] - m["datetime"]) <= H).to_numpy().astype(np.int8)

def labels_bruteforce(tele_sorted, failures_df, n=3000, seed=SEED):
    """Independent path 2: direct definition on a random sample of rows."""
    rng = np.random.default_rng(seed)
    rows = rng.choice(len(tele_sorted), size=n, replace=False)
    fl = {m: g["datetime"].to_numpy() for m, g in failures_df.groupby("machineID")}
    out = []
    for r in rows:
        m, t = tele_sorted.at[r, "machineID"], tele_sorted.at[r, "datetime"].to_datetime64()
        ft = fl.get(m, np.array([], dtype="datetime64[ns]"))
        out.append(int(((ft > t) & (ft <= t + H64)).any()))
    return rows, np.array(out, dtype=np.int8)

lab_primary = labels_searchsorted(tele, failures)
lab_asof = labels_merge_asof(tele, failures)
rows_bf, lab_bf = labels_bruteforce(tele, failures)

record("L-1 label identical across three independent implementations",
       np.array_equal(lab_primary, lab_asof) and np.array_equal(lab_primary[rows_bf], lab_bf),
       f"searchsorted == merge_asof on {len(tele):,} rows; == brute force on {len(rows_bf):,} sampled rows")

tele[TARGET] = lab_primary

# Boundary handling: keep only rows whose complete 24 h future window lies inside the observation period
last_ts = tele.groupby("machineID")["datetime"].transform("max")
tele["observable"] = tele["datetime"] <= (last_ts - H)
n_dropped = int((~tele["observable"]).sum())
record("L-2 last 24 rows of every machine dropped (unobservable horizon)",
       (tele.groupby("machineID")["observable"].apply(lambda s: int((~s).sum())) == HORIZON_HOURS).all(),
       f"{n_dropped:,} rows dropped in total")

obs = tele[tele["observable"]]
pos, tot = int(obs[TARGET].sum()), len(obs)
print(f"\nCandidate rows: {tot:,} | positives: {pos:,} | negatives: {tot - pos:,} | positive rate: {100 * pos / tot:.4f}%")
print("Note: positives are NOT independent — one failure produces up to 24 consecutive positive rows.")

## 5. Simulated federated clients (factories)

The primary manifest has five clients with 20 machines each. Fixed per-model quotas create a deliberately non-IID machine-model distribution while ensuring that `factory_5` contains multiple models; this reduces, but does not eliminate, confounding between site and machine type. A second manifest preserves the stronger model-blocked shift scenario as a separate stress-test setup. Both are deterministic and versioned under `prep_v3/manifests/`.


In [ ]:
# 5.1  Deterministic mixed-model primary manifest + model-blocked stress manifest
FACTORY_ORDER = [f"factory_{i}" for i in range(1, N_FACTORIES + 1)]

# Fixed Azure dataset quotas. Column totals must match the dataset; row totals are 20.
# This creates non-IID model mixtures without making the primary held-out client a single-model client.
MODEL_QUOTAS = {
    "factory_1": {"model1": 8, "model2": 4, "model3": 4, "model4": 4},
    "factory_2": {"model1": 4, "model2": 7, "model3": 5, "model4": 4},
    "factory_3": {"model1": 2, "model2": 3, "model3": 10, "model4": 5},
    "factory_4": {"model1": 1, "model2": 2, "model3": 8, "model4": 9},
    "factory_5": {"model1": 1, "model2": 1, "model3": 8, "model4": 10},
}

observed_model_counts = machines["model"].value_counts().to_dict()
quota_model_counts = {
    model: sum(MODEL_QUOTAS[client].get(model, 0) for client in FACTORY_ORDER)
    for model in sorted({m for q in MODEL_QUOTAS.values() for m in q})
}
record("C-0 fixed partition quotas match the raw machine-model counts",
       observed_model_counts == quota_model_counts
       and all(sum(MODEL_QUOTAS[c].values()) == MACHINES_PER_FACTORY for c in FACTORY_ORDER),
       f"observed={observed_model_counts}; quota totals={quota_model_counts}")

assigned_parts = []
for j, model_name in enumerate(sorted(quota_model_counts)):
    model_rows = (machines[machines["model"] == model_name]
                  .sort_values(["age", "machineID"])
                  .sample(frac=1, random_state=SEED + j)
                  .reset_index(drop=True))
    cursor = 0
    for client in FACTORY_ORDER:
        n_take = MODEL_QUOTAS[client].get(model_name, 0)
        part = model_rows.iloc[cursor:cursor + n_take].copy()
        part["client_id"] = client
        assigned_parts.append(part)
        cursor += n_take
    assert cursor == len(model_rows), f"Quota allocation incomplete for {model_name}"

ordered = pd.concat(assigned_parts, ignore_index=True)
manifest = (ordered[["machineID", "client_id", "model", "age"]]
            .sort_values(["client_id", "machineID"]).reset_index(drop=True))
composition = pd.crosstab(manifest["client_id"], manifest["model"]).reindex(
    index=FACTORY_ORDER, columns=sorted(quota_model_counts), fill_value=0)
composition["mean_age"] = manifest.groupby("client_id")["age"].mean().reindex(FACTORY_ORDER).round(1)

record("C-1 five mixed-model factories x 20 machines; each machine assigned once",
       manifest["machineID"].is_unique
       and (manifest.groupby("client_id").size() == MACHINES_PER_FACTORY).all()
       and manifest["client_id"].nunique() == N_FACTORIES
       and (composition[sorted(quota_model_counts)].gt(0).sum(axis=1) >= 2).all()
       and (composition.loc[HELD_OUT, sorted(quota_model_counts)] > 0).sum() >= 3,
       "primary clients contain multiple models; factory_5 has at least three model types")

# Preserve an alternative severe machine-type shift: contiguous groups after sorting by model.
stress_order = machines.sort_values(["model", "age", "machineID"]).reset_index(drop=True).copy()
stress_order["client_id"] = [f"factory_{i // MACHINES_PER_FACTORY + 1}" for i in range(len(stress_order))]
stress_manifest = (stress_order[["machineID", "client_id", "model", "age"]]
                   .sort_values(["client_id", "machineID"]).reset_index(drop=True))

manifest.to_csv(MAN_DIR / "factory_manifest.csv", index=False)
manifest.to_json(MAN_DIR / "factory_manifest.json", orient="records", indent=2)
stress_manifest.to_csv(MAN_DIR / "factory_manifest_type_shift_stress.csv", index=False)

# Compare against the prior manifest for audit, but do not fail merely because the primary
# partition intentionally changed. The old file is never overwritten.
legacy_path = PROJECT_DIR / "manifests" / "factory_manifest.csv"
legacy_comparison = {"legacy_path": str(legacy_path), "legacy_found": legacy_path.exists(), "same_mapping": None}
if legacy_path.exists():
    legacy = pd.read_csv(legacy_path)[["machineID", "client_id"]]
    active_map = manifest[["machineID", "client_id"]]
    cmp_ = active_map.merge(legacy, on="machineID", suffixes=("", "_legacy"), validate="one_to_one")
    legacy_comparison["same_mapping"] = bool(len(cmp_) == 100 and (cmp_["client_id"] == cmp_["client_id_legacy"]).all())
    print("Prior manifest found:", legacy_path)
    print("Same machine-to-client mapping:", legacy_comparison["same_mapping"])
    print("The revised primary mapping is versioned in prep_v3; prior mapping remains untouched.")
else:
    print("No root-level legacy manifest found; prep_v3 manifest created.")
(MAN_DIR / "legacy_manifest_comparison.json").write_text(json.dumps(legacy_comparison, indent=2))
(CFG_DIR / "factory_partition_config.json").write_text(json.dumps({
    "primary_method": "fixed non-IID model quotas; deterministic shuffle within each model",
    "seed": SEED,
    "model_quotas": MODEL_QUOTAS,
    "observed_model_counts": observed_model_counts,
    "primary_manifest": "manifests/factory_manifest.csv",
    "alternate_stress_manifest": "manifests/factory_manifest_type_shift_stress.csv",
    "alternate_stress_note": "Model-blocked partition; not used in the primary splits in this notebook. Run as a separate experiment.",
}, indent=2))

print("\nPrimary machine-model composition per factory:")
print(composition.to_string())
print("\nStress-test model composition (model-blocked assignment):")
print(pd.crosstab(stress_manifest["client_id"], stress_manifest["model"]).to_string())
print("\nSaved primary manifest and alternate model-type-shift stress manifest under", MAN_DIR)

## 6. Splits: temporal windows, held-out factory, and calibration isolation

- **Development factories 1–4:** `train` → 24 h purge gap → `validation` → 24 h purge gap → `test_time_late`.
- **Held-out factory 5:** no supervised rows enter train/validation/tuning. Its first 50% of the timeline is a feature-only, unlabeled calibration window for the optional `local` scaler. This file excludes labels and is not an evaluation split.
- `test_site_full` contains all held-out-factory rows from the calibration cutoff onward, so no evaluation row is used to fit `local_factory_5`.
- `test_site_late` is the late-window subset of `test_site_full`; it is stricter, but the two tests are nested and are not statistically independent.
- `test_time_late` uses factories 1–4 in the same late window.

The primary factory assignment contains all machine types in every client, with intentionally different proportions. The separate model-blocked manifest is a stress-test alternative, not evidence of site-only generalization by itself.


In [ ]:
# 6.1  Build splits (metadata + labels only); held-out calibration keys are kept label-free
cand = (tele.loc[tele["observable"], ["datetime", "machineID", TARGET]]
        .merge(manifest[["machineID", "client_id"]], on="machineID", how="left", validate="many_to_one")
        .sort_values(["machineID", "datetime"]).reset_index(drop=True))
assert cand["client_id"].notna().all()

uts = np.sort(cand["datetime"].unique())
VAL_START = pd.Timestamp(uts[int(len(uts) * TRAIN_FRACTION)])
TRAIN_CUTOFF = VAL_START - H
remainder = uts[uts >= VAL_START.to_datetime64()]
TEST_START = pd.Timestamp(remainder[int(len(remainder) * VAL_FRACTION_OF_REMAINDER)])
VAL_CUTOFF = TEST_START - H
CALIBRATION_CUTOFF = pd.Timestamp(uts[int(len(uts) * HELD_OUT_CALIBRATION_FRACTION)])
assert CALIBRATION_CUTOFF + H < TRAIN_CUTOFF, "Calibration must end well before the development training cutoff"

dt = cand["datetime"]
is_dev = cand["client_id"].isin(TRAINING_FACTORIES)
is_ho = cand["client_id"].eq(HELD_OUT)
MASKS = {
    "train":          is_dev & (dt < TRAIN_CUTOFF),
    "validation":     is_dev & (dt >= VAL_START) & (dt < VAL_CUTOFF),
    "test_time_late": is_dev & (dt >= TEST_START),
    "test_site_late": is_ho & (dt >= TEST_START),
    "test_site_full": is_ho & (dt >= CALIBRATION_CUTOFF),
}
SPLIT_KEYS = {name: cand.loc[m].reset_index(drop=True) for name, m in MASKS.items()}
SPLIT_NAMES = list(SPLIT_KEYS)
# This calibration table intentionally omits TARGET to make label use impossible through this artifact.
CALIBRATION_KEYS = (cand.loc[is_ho & (dt < CALIBRATION_CUTOFF), ["datetime", "machineID", "client_id"]]
                    .reset_index(drop=True))

print("Time boundaries")
print(f"  train             : {SPLIT_KEYS['train']['datetime'].min()} -> {SPLIT_KEYS['train']['datetime'].max()}")
print(f"  validation        : {SPLIT_KEYS['validation']['datetime'].min()} -> {SPLIT_KEYS['validation']['datetime'].max()}")
print(f"  calibration cutoff: {CALIBRATION_CUTOFF} (fraction={HELD_OUT_CALIBRATION_FRACTION:.0%})")
print(f"  late test window  : {TEST_START} -> {cand['datetime'].max()}")

# --- temporal and site isolation verification
tr, va = SPLIT_KEYS["train"], SPLIT_KEYS["validation"]
record("S-1 24h purge gaps: no label window crosses train/validation/test boundary",
       tr["datetime"].max() + H < VAL_START and va["datetime"].max() + H < TEST_START,
       f"train end + 24h={tr['datetime'].max() + H} < val start={VAL_START}; "
       f"validation end + 24h={va['datetime'].max() + H} < test start={TEST_START}")

ho_machines = set(manifest.loc[manifest["client_id"] == HELD_OUT, "machineID"])
dev_machines = set(manifest.loc[manifest["client_id"].isin(TRAINING_FACTORIES), "machineID"])
record("S-2 held-out machines never appear in supervised development splits",
       ho_machines.isdisjoint(dev_machines)
       and all(HELD_OUT not in set(SPLIT_KEYS[s]["client_id"]) for s in ["train", "validation", "test_time_late"])
       and all(set(SPLIT_KEYS[s]["client_id"]) == {HELD_OUT} for s in ["test_site_late", "test_site_full"]),
       f"{len(ho_machines)} held-out machines isolated from train/validation/time-test rows")

record("S-3 no duplicate (machine, timestamp) rows inside any evaluation split",
       all(not d.duplicated(["machineID", "datetime"]).any() for d in SPLIT_KEYS.values())
       and cand.groupby("machineID")["client_id"].nunique().max() == 1)

late_in_full = SPLIT_KEYS["test_site_late"].merge(
    SPLIT_KEYS["test_site_full"], on=["machineID", "datetime"], how="left", indicator=True)
record("S-4 late site test is a subset of post-calibration site test; all evaluation splits contain positives",
       bool((late_in_full["_merge"] == "both").all())
       and all(d[TARGET].sum() > 0 for d in SPLIT_KEYS.values()),
       f"test_site_full rows={len(SPLIT_KEYS['test_site_full']):,}; late subset rows={len(SPLIT_KEYS['test_site_late']):,}")

calib_eval_overlap = CALIBRATION_KEYS[["machineID", "datetime"]].merge(
    SPLIT_KEYS["test_site_full"][["machineID", "datetime"]], on=["machineID", "datetime"], how="inner")
record("S-5 held-out calibration is earlier than and disjoint from all site-evaluation rows",
       len(CALIBRATION_KEYS) > 0 and calib_eval_overlap.empty
       and CALIBRATION_KEYS["datetime"].max() < SPLIT_KEYS["test_site_full"]["datetime"].min()
       and CALIBRATION_KEYS["datetime"].max() + H < TRAIN_CUTOFF,
       f"calibration rows={len(CALIBRATION_KEYS):,}; eval starts={SPLIT_KEYS['test_site_full']['datetime'].min()}")

# Informational continuity with the old version. Differences are expected because the primary
# factory assignment and site-test definition have been intentionally revised.
print("\nPrevious-v2 counts are not used as acceptance criteria; partition and held-out evaluation changed intentionally.")

### 6.2 How much evidence does each split contain?

Rows are not independent: one failure creates up to 24 positive rows. The real sample size is the number of **distinct failure events** covered by positive labels. Metrics (PR-AUC, recall, F1) computed on splits with few events are statistically fragile — report confidence intervals and prefer the splits with more events.

In [ ]:
# 6.2  Rows, positives and distinct failure events per split and per factory
def covered_events(df):
    """Distinct failure events (machine, timestamp) having >=1 row t in the split with t < f <= t + 24h."""
    ts = {m: np.sort(g["datetime"].to_numpy()) for m, g in df.groupby("machineID")}
    n = 0
    for m, g in fail_events.groupby("machineID"):
        if m not in ts:
            continue
        f = g["datetime"].to_numpy()
        lo = np.searchsorted(ts[m], f - H64, side="left")      # first row with t >= f - 24h
        hi = np.searchsorted(ts[m], f, side="left")            # first row with t >= f  (rows before are t < f)
        n += int(((hi - lo) > 0).sum())
    return n

rows = []
for split, df in SPLIT_KEYS.items():
    for client, g in list(df.groupby("client_id")) + [("ALL", df)]:
        ev = covered_events(g)
        rows.append({"split": split, "factory": client, "rows": len(g), "machines": g["machineID"].nunique(),
                     "positive_rows": int(g[TARGET].sum()), "positive_rate_pct": round(100 * g[TARGET].mean(), 3),
                     "failure_events": ev, "fragile(<%d events)" % FRAGILE_EVENT_THRESHOLD: ev < FRAGILE_EVENT_THRESHOLD})
EVENT_TABLE = pd.DataFrame(rows)
EVENT_TABLE.to_csv(REP_DIR / "split_event_table.csv", index=False)
print(EVENT_TABLE.to_string(index=False))
n_frag = int(EVENT_TABLE.iloc[:, -1].sum())
print(f"\n{n_frag} of {len(EVENT_TABLE)} split/factory cells have fewer than {FRAGILE_EVENT_THRESHOLD} distinct failure events — "
      "per-factory and late-window metrics there are statistically fragile; use confidence intervals.")
record("S-5 event table produced", len(EVENT_TABLE) > 0,
       f"{len(EVENT_TABLE)} rows saved to split_event_table.csv")

## 7. Maintenance records: time semantics (D2)

Maintenance records are only allowed as features if their timing is understood. This cell measures how often a maintenance (component replacement) record coincides with a failure of the same machine.

In [ ]:
# 7.1  Time semantics of maintenance records relative to failures
maint_k = maint.rename(columns={"comp": "component"})
fail_k = failures.rename(columns={"failure": "component"})

at_failure_any = maint_k.merge(fail_events, on=["machineID", "datetime"], how="inner")
at_failure_same = maint_k.merge(fail_k, on=["machineID", "datetime", "component"], how="inner")
fails_with_replacement = (fail_k.merge(maint_k, on=["machineID", "datetime", "component"], how="left", indicator=True)["_merge"]
                          .eq("both").mean())

MAINT_SEMANTICS = {
    "maintenance_records": len(maint),
    "records_before_telemetry_start": int((maint["datetime"] < tele["datetime"].min()).sum()),
    "records_at_a_failure_timestamp_any_component": len(at_failure_any),
    "records_at_a_failure_timestamp_same_component": len(at_failure_same),
    "share_of_failures_with_same_timestamp_replacement": round(float(fails_with_replacement), 4),
}
for k, v in MAINT_SEMANTICS.items():
    print(f"  {k:55s} {v}")

print("""
Interpretation
  * Maintenance records that coincide with a failure are reactive replacements. After such a failure,
    hours_since_last_maintenance resets, so the feature partly encodes 'time since the last failure'.
  * The feature uses only events STRICTLY BEFORE the prediction time, so rows that precede a failure
    never see that failure's replacement record: no label leakage.
  * Decision: keep maintenance recency in the primary feature set 'full' (information available in
    deployment) and ALSO save the feature set 'no_maintenance'; the later ablation study must report both.""")
record("D-2 maintenance time semantics measured and documented", True,
       f"{MAINT_SEMANTICS['share_of_failures_with_same_timestamp_replacement']:.1%} of failure rows have a same-timestamp replacement of the same component")

## 8. Past-only feature engineering

Five feature groups, every one computed from information available **at or before** the prediction time `t`:

| Group | Features |
|---|---|
| `sensor_raw` | `volt`, `rotate`, `pressure`, `vibration` at time `t` |
| `rolling_stats` | mean and std of each sensor over the previous 24 h and 7 d (the window **excludes** row `t`) |
| `error_recency` | `log1p(hours since the last error strictly before t)`, `has_prior_error` |
| `maintenance_recency` | `log1p(hours since the last maintenance record strictly before t)` |
| `history_counts` | number of past hourly observations available, capped at 24 and 168 |

Missing-history rules (only the sentinel is learned, and only from training rows): early rolling mean → the current reading; early rolling std → 0; "no prior error" → training-only sentinel (max observed training recency + 1 h) plus the `has_prior_error` indicator. Recency features are `log1p`-transformed to remove their heavy tail before scaling.

In [ ]:
# 8.1  Feature definitions (pure functions; reused by the leakage tests)
FEATURE_GROUPS = {
    "sensor_raw": list(SENSORS),
    "rolling_stats": [f"{s}_past_{stat}_{w}" for s in SENSORS for w in ROLLING_WINDOWS for stat in ("mean", "std")],
    "error_recency": ["hours_since_last_error_log1p", "has_prior_error"],
    "maintenance_recency": ["hours_since_last_maintenance_log1p"],
    "history_counts": [f"past_observation_count_{w}" for w in ROLLING_WINDOWS],
}
ALL_FEATURES = [c for g in FEATURE_GROUPS.values() for c in g]
NO_MAINT_FEATURES = [c for c in ALL_FEATURES if c not in FEATURE_GROUPS["maintenance_recency"]]
META_COLS = ["datetime", "machineID", "client_id"]
assert len(ALL_FEATURES) == 25 and len(set(ALL_FEATURES)) == 25

def _hours_since_last_event(keys, events):
    """Hours since the most recent event of the same machine STRICTLY BEFORE each row's timestamp (NaN if none)."""
    ev = events[["machineID", "datetime"]].rename(columns={"datetime": "event_time"})
    ev["datetime"] = ev["event_time"]
    ev = ev.sort_values("datetime")
    left = keys[["machineID", "datetime"]].copy()
    left["_row"] = np.arange(len(left))
    left = left.sort_values("datetime")
    m = pd.merge_asof(left, ev, on="datetime", by="machineID", direction="backward", allow_exact_matches=False)
    m = m.sort_values("_row")
    return ((m["datetime"] - m["event_time"]) / pd.Timedelta(hours=1)).to_numpy(dtype="float64")

def build_raw_features(tele_df, errors_df, maint_df):
    """Past-only features before any train-derived filling. Returns NaN where history is missing."""
    df = tele_df[["machineID", "datetime"] + SENSORS].sort_values(["machineID", "datetime"]).reset_index(drop=True)
    out = df.copy()
    for s in SENSORS:
        past = df.groupby("machineID", sort=False)[s].shift(1)              # strictly earlier reading
        grp = past.groupby(df["machineID"], sort=False)
        for wname, w in ROLLING_WINDOWS.items():
            out[f"{s}_past_mean_{wname}"] = grp.rolling(w, min_periods=1).mean().reset_index(level=0, drop=True)
            out[f"{s}_past_std_{wname}"] = grp.rolling(w, min_periods=2).std(ddof=0).reset_index(level=0, drop=True)
    pos = df.groupby("machineID", sort=False).cumcount().to_numpy()
    for wname, w in ROLLING_WINDOWS.items():
        out[f"past_observation_count_{wname}"] = np.minimum(pos, w).astype("int16")
    out["hours_since_last_error_raw"] = _hours_since_last_event(df, errors_df)
    out["hours_since_last_maintenance_raw"] = _hours_since_last_event(df, maint_df)
    return out

def finalize_features(raw, error_sentinel_hours):
    """Apply the missing-history rules and log1p; returns keys + the 25 model features (float32)."""
    out = raw[["machineID", "datetime"]].copy()
    for s in SENSORS:
        out[s] = raw[s]
        for wname in ROLLING_WINDOWS:
            out[f"{s}_past_mean_{wname}"] = raw[f"{s}_past_mean_{wname}"].fillna(raw[s])      # current reading
            out[f"{s}_past_std_{wname}"] = raw[f"{s}_past_std_{wname}"].fillna(0.0)
    for wname in ROLLING_WINDOWS:
        out[f"past_observation_count_{wname}"] = raw[f"past_observation_count_{wname}"]
    e = raw["hours_since_last_error_raw"]
    out["has_prior_error"] = e.notna().astype("int8")
    out["hours_since_last_error_log1p"] = np.log1p(e.fillna(error_sentinel_hours))
    mt = raw["hours_since_last_maintenance_raw"]
    if mt.isna().any():
        raise ValueError("Unexpected rows without a prior maintenance record")
    out["hours_since_last_maintenance_log1p"] = np.log1p(mt)
    out[ALL_FEATURES] = out[ALL_FEATURES].astype("float32")
    return out[["machineID", "datetime"] + ALL_FEATURES]

print("Feature groups:", {g: len(c) for g, c in FEATURE_GROUPS.items()}, "| total:", len(ALL_FEATURES))

In [ ]:
# 8.2  Build features, derive the training-only sentinel, save evaluation splits and feature-only calibration data
raw_feat = build_raw_features(tele, errors, maint)
assert len(raw_feat) == len(tele)

# has_prior_maintenance would be constant (maintenance logs start before telemetry) -> excluded, verified here
record("FE-0 every telemetry row has a prior maintenance record (has_prior_maintenance is constant, excluded)",
       bool(raw_feat["hours_since_last_maintenance_raw"].notna().all()))

# Training-only sentinel for 'no prior error'
tr_err = SPLIT_KEYS["train"][["machineID", "datetime"]].merge(
    raw_feat[["machineID", "datetime", "hours_since_last_error_raw"]], on=["machineID", "datetime"], how="left", validate="one_to_one")
ERROR_SENTINEL_HOURS = float(tr_err["hours_since_last_error_raw"].dropna().max()) + 1.0
print(f"Training-only error-recency sentinel: {ERROR_SENTINEL_HOURS:.1f} h  (log1p -> {np.log1p(ERROR_SENTINEL_HOURS):.3f})")

feat_all = finalize_features(raw_feat, ERROR_SENTINEL_HOURS)

SPLIT_DF, fill_rows = {}, []
for name, keys in SPLIT_KEYS.items():
    df = keys.merge(feat_all, on=["machineID", "datetime"], how="left", validate="one_to_one")
    raw_part = keys[["machineID", "datetime"]].merge(raw_feat, on=["machineID", "datetime"], how="left", validate="one_to_one")
    mean_cols = [c for c in raw_part.columns if "_past_mean_" in c]; std_cols = [c for c in raw_part.columns if "_past_std_" in c]
    fill_rows.append({"split": name, "rows": len(df),
                      "rows_with_filled_rolling_mean": int(raw_part[mean_cols].isna().any(axis=1).sum()),
                      "rows_with_filled_rolling_std": int(raw_part[std_cols].isna().any(axis=1).sum()),
                      "rows_with_no_prior_error": int(raw_part["hours_since_last_error_raw"].isna().sum())})
    assert len(df) == len(keys), name
    assert np.isfinite(df[ALL_FEATURES].to_numpy()).all(), f"non-finite feature in {name}"
    df = df[META_COLS + [TARGET] + ALL_FEATURES]
    df.to_parquet(PROC_DIR / f"{name}.parquet", index=False)
    SPLIT_DF[name] = df

# Persist held-out calibration as feature-only data: no TARGET/label column is written.
calibration_df = CALIBRATION_KEYS.merge(
    feat_all, on=["machineID", "datetime"], how="left", validate="one_to_one")
calibration_df = calibration_df[META_COLS + ALL_FEATURES]
assert TARGET not in calibration_df.columns and np.isfinite(calibration_df[ALL_FEATURES].to_numpy()).all()
calibration_df.to_parquet(PROC_DIR / "heldout_calibration_features.parquet", index=False)

FILL_TABLE = pd.DataFrame(fill_rows)
print("\nRows affected by the missing-history rules:")
print(FILL_TABLE.to_string(index=False))
print(f"\nSaved feature-only held-out calibration rows: {len(calibration_df):,}; labels are absent.")

train_df = SPLIT_DF["train"]
constant = [c for c in ALL_FEATURES if train_df[c].nunique() <= 1]
record("FE-1 all 25 features finite, none constant in training", len(constant) == 0 and len(ALL_FEATURES) == 25,
       f"constant features: {constant or 'none'}")

FEATURE_CONFIG = {
    "groups": FEATURE_GROUPS,
    "feature_sets": {"full": ALL_FEATURES, "no_maintenance": NO_MAINT_FEATURES},
    "primary_feature_set": "full", "ablation_feature_set": "no_maintenance",
    "excluded_constant_features": ["has_prior_maintenance"],
    "identifier_columns": META_COLS, "target": TARGET,
    "error_recency_sentinel_hours": ERROR_SENTINEL_HOURS,
    "missing_history_rules": {"rolling_mean": "current sensor reading", "rolling_std": 0.0,
                              "no_prior_error": "training-only sentinel + has_prior_error indicator",
                              "recency": "log1p", "rolling_windows_rows": ROLLING_WINDOWS},
    "boundaries": {"train_cutoff": str(TRAIN_CUTOFF), "val_start": str(VAL_START), "val_cutoff": str(VAL_CUTOFF),
                   "test_start": str(TEST_START), "held_out_calibration_cutoff": str(CALIBRATION_CUTOFF),
                   "held_out_calibration_fraction": HELD_OUT_CALIBRATION_FRACTION,
                   "horizon_hours": HORIZON_HOURS},
}
(CFG_DIR / "feature_groups.json").write_text(json.dumps(FEATURE_CONFIG, indent=2))
FILL_TABLE.to_csv(REP_DIR / "missing_history_fill_counts.csv", index=False)
print("\nSaved:", ", ".join(f"{n}.parquet" for n in SPLIT_DF), "+ heldout_calibration_features.parquet + configs/feature_groups.json")

## 9. Normalization policies

Both policies are fitted on the allowed rows and saved. The held-out local scaler uses only the earlier, feature-only calibration file; the calibration file has no target column and is disjoint from `test_site_full` and `test_site_late`.

| Policy | Scalers | Held-out factory |
|---|---|---|
| `global` | One scaler fitted on pooled supervised training rows from factories 1–4 | Same global scaler; no feature rows from factory 5 used for fitting |
| `local` | One scaler per development factory fitted on that factory's training rows | `local_factory_5` fitted on `heldout_calibration_features.parquet`, containing only features and metadata before `CALIBRATION_CUTOFF` |

This local policy represents **unsupervised site adaptation**, not zero-shot normalization. `test_site_full` begins at the calibration cutoff, so evaluation rows do not participate in fitting. `test_site_late` is nested within `test_site_full` and must not be treated as an independent sample.


In [ ]:
# 9.1  Fit and save scalers on their explicitly allowed source rows
SPLIT_DF = {n: pd.read_parquet(PROC_DIR / f"{n}.parquet") for n in SPLIT_NAMES}
train_df = SPLIT_DF["train"]
calib = pd.read_parquet(PROC_DIR / "heldout_calibration_features.parquet")

record("N-0 calibration artifact contains no target and no evaluation rows",
       TARGET not in calib.columns
       and list(calib.columns) == META_COLS + ALL_FEATURES
       and not calib.duplicated(["machineID", "datetime"]).any()
       and calib["datetime"].max() < SPLIT_DF["test_site_full"]["datetime"].min(),
       f"{len(calib):,} feature-only calibration rows; evaluation starts {SPLIT_DF['test_site_full']['datetime'].min()}")

def fit_scaler(df):
    sc = StandardScaler().fit(df[ALL_FEATURES].to_numpy(dtype="float64"))
    assert (sc.var_ > 0).all(), "zero-variance feature"
    return sc

SCALERS, SCALER_SOURCES = {}, {}
SCALERS["global"] = fit_scaler(train_df)
SCALER_SOURCES["global"] = f"train rows of {TRAINING_FACTORIES}"
for f in TRAINING_FACTORIES:
    SCALERS[f"local_{f}"] = fit_scaler(train_df[train_df["client_id"] == f])
    SCALER_SOURCES[f"local_{f}"] = f"train rows of {f} only"

SCALERS[f"local_{HELD_OUT}"] = fit_scaler(calib)
SCALER_SOURCES[f"local_{HELD_OUT}"] = (
    f"feature-only unlabeled {HELD_OUT} calibration rows with datetime < {CALIBRATION_CUTOFF}")

stats = {}
for name, sc in SCALERS.items():
    joblib.dump(sc, SCALER_DIR / f"{name}.joblib")
    stats[name] = {"fit_source": SCALER_SOURCES[name], "n_rows": int(sc.n_samples_seen_),
                   "features": ALL_FEATURES, "mean": sc.mean_.tolist(), "scale": sc.scale_.tolist()}
(CFG_DIR / "scaler_statistics.json").write_text(json.dumps(stats, indent=2))

PROTOCOL = {
    "method": "StandardScaler (per column), fitted in float64 on saved float32 feature values",
    "policies": {
        "global": "one scaler on pooled training rows of factories 1-4; applied to every evaluation split",
        "local": "one scaler per client; held-out factory uses feature-only unlabeled rows before CALIBRATION_CUTOFF",
    },
    "held_out_calibration": {
        "file": "processed/heldout_calibration_features.parquet",
        "cutoff": str(CALIBRATION_CUTOFF),
        "fraction_of_timeline": HELD_OUT_CALIBRATION_FRACTION,
        "contains_target": False,
        "evaluation_disjoint": True,
    },
    "never_fitted_on": ["validation rows for supervised clients", "evaluation rows", "any evaluation labels"],
    "note_test_site_full": "Contains only held-out rows at/after CALIBRATION_CUTOFF; local scaler calibration rows are strictly earlier and saved without labels.",
    "feature_set_selection": "select columns after scaling (feature_sets in feature_groups.json)",
}
(CFG_DIR / "preprocessing_protocol.json").write_text(json.dumps(PROTOCOL, indent=2))

summary = pd.DataFrame([{"scaler": n, "rows": s["n_rows"], "fit_source": s["fit_source"][:90]} for n, s in stats.items()])
print(summary.to_string(index=False))
record("N-1 six scalers fitted from allowed sources and saved", len(SCALERS) == 1 + N_FACTORIES,
       "global + 4 local training factories + local held-out feature-only calibration")

## 10. Loader for the next phase

This cell writes `prep_v3/fedkg_data.py`, a small loader used by every later notebook, so that all experiments read the data and apply the normalization policy in exactly the same way, then tests it.

```python
import sys; sys.path.insert(0, "/content/drive/MyDrive/FedKG-Maint/prep_v3")
from fedkg_data import load_split
X, y, meta = load_split("train", policy="global", feature_set="full")
```

In [ ]:
# 10.1  Write and test the loader module
MODULE_SRC = '''"""Loader for the prepared FedKG-Maint data (v3).

    import sys; sys.path.insert(0, "<PROJECT_DIR>/prep_v3")
    from fedkg_data import load_split
    X, y, meta = load_split("train", policy="global", feature_set="full")
"""
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parent
SPLITS = ("train", "validation", "test_time_late", "test_site_late", "test_site_full")
META_COLS = ["datetime", "machineID", "client_id"]
TARGET = "label_failure_24h"


def load_split(split, policy="global", feature_set="full", root=ROOT):
    """Return (X float32, y float32, meta DataFrame).

    policy:      "global" | "local" | "none"  (normalization policy, see configs/preprocessing_protocol.json)
    feature_set: "full" (25 features) | "no_maintenance" (24 features)
    """
    root = Path(root)
    cfg = json.loads((root / "configs" / "feature_groups.json").read_text())
    all_cols = cfg["feature_sets"]["full"]
    keep = [all_cols.index(c) for c in cfg["feature_sets"][feature_set]]
    df = pd.read_parquet(root / "processed" / f"{split}.parquet")
    X = df[all_cols].to_numpy(dtype="float64")
    if policy == "global":
        X = joblib.load(root / "scalers" / "global.joblib").transform(X)
    elif policy == "local":
        scaled = np.empty_like(X)
        for client in df["client_id"].unique():
            m = (df["client_id"] == client).to_numpy()
            scaled[m] = joblib.load(root / "scalers" / f"local_{client}.joblib").transform(X[m])
        X = scaled
    elif policy != "none":
        raise ValueError("policy must be 'global', 'local' or 'none'")
    return X[:, keep].astype("float32"), df[TARGET].to_numpy("float32"), df[META_COLS].reset_index(drop=True)


def load_client(split, client_id, **kwargs):
    """Rows of one factory only (what a federated client would hold)."""
    X, y, meta = load_split(split, **kwargs)
    m = (meta["client_id"] == client_id).to_numpy()
    return X[m], y[m], meta[m].reset_index(drop=True)
'''
(PREP_DIR / "fedkg_data.py").write_text(MODULE_SRC)

import importlib
sys.path.insert(0, str(PREP_DIR))
import fedkg_data
importlib.reload(fedkg_data)

report = []
for split in fedkg_data.SPLITS:
    for policy in ("global", "local", "none"):
        for fs, n_feat in (("full", 25), ("no_maintenance", 24)):
            X, y, meta = fedkg_data.load_split(split, policy=policy, feature_set=fs)
            assert X.shape == (len(y), n_feat) and np.isfinite(X).all() and len(meta) == len(y)
    report.append((split, X.shape[0]))

# the scaled training data must have mean 0 / std 1 (global) and per factory (local)
Xg, _, _ = fedkg_data.load_split("train", "global")
Xl, _, ml = fedkg_data.load_split("train", "local")
ok_global = np.allclose(Xg.mean(0), 0, atol=1e-3) and np.allclose(Xg.std(0), 1, atol=1e-3)
ok_local = all(np.allclose(Xl[(ml["client_id"] == f).to_numpy()].mean(0), 0, atol=1e-3) for f in TRAINING_FACTORIES)
Xc, yc, mc = fedkg_data.load_client("train", "factory_1", policy="local")
record("N-2 loader works for every split x policy x feature set; scaling behaves as specified",
       ok_global and ok_local and set(mc["client_id"]) == {"factory_1"}, f"{dict(report)}")

## 11. Leakage and integrity tests

- **F1 — future-perturbation:** for sampled rows from every evaluation split, changing sensor readings after `t` and moving error/maintenance events at or after `t` must leave the feature vector at `t` unchanged. A positive control perturbs the past and must change the features.
- **F2 — scaler source:** JSON and `.joblib` scaler statistics are recomputed from their permitted source rows. The held-out scaler source is the feature-only calibration artifact, not an evaluation split.
- **F3 — evaluation separation:** the held-out calibration artifact contains no target column and has no machine/timestamp keys in common with `test_site_full`.
- **F4 — schema:** all evaluation splits have identical columns, no identifiers or target appear among model features, and the feature groups match the saved config.


In [ ]:
# 11.1  Leakage and integrity tests
rng = np.random.default_rng(SEED)
N_PER_SPLIT, N_CONTROL = 200, 10

tele_m = {m: g[["machineID", "datetime"] + SENSORS].reset_index(drop=True) for m, g in tele.groupby("machineID")}
err_m = {m: g[["machineID", "datetime"]].reset_index(drop=True) for m, g in errors.groupby("machineID")}
mnt_m = {m: g[["machineID", "datetime"]].reset_index(drop=True) for m, g in maint.groupby("machineID")}
EMPTY = pd.DataFrame({"machineID": pd.Series(dtype="int64"), "datetime": pd.Series(dtype="datetime64[ns]")})

def recompute_row(split, i, mode):
    """Recompute the 25 features of row i after perturbing the future ('future') or the past ('past')."""
    row = SPLIT_DF[split].iloc[i]
    m, t = int(row["machineID"]), row["datetime"]
    tl, er, mt = tele_m[m].copy(), err_m.get(m, EMPTY).copy(), mnt_m.get(m, EMPTY).copy()
    if mode == "future":
        f = (tl["datetime"] > t).to_numpy()
        tl.loc[f, SENSORS] = rng.normal(1000, 500, size=(int(f.sum()), len(SENSORS)))
        for ev in (er, mt):
            g = (ev["datetime"] >= t).to_numpy()
            new_times = (t + pd.to_timedelta(rng.integers(0, 3000, int(g.sum())), unit="h")).astype("datetime64[ns]")
            ev.loc[g, "datetime"] = new_times.to_numpy()
        extra = pd.DataFrame({"machineID": [m, m], "datetime": pd.to_datetime([t, t + pd.Timedelta(hours=1)]).astype("datetime64[ns]")})
        er, mt = pd.concat([er, extra], ignore_index=True), pd.concat([mt, extra], ignore_index=True)
    else:  # past perturbation (positive control)
        p = ((tl["datetime"] < t) & (tl["datetime"] >= t - pd.Timedelta(hours=48))).to_numpy()
        tl.loc[p, SENSORS] = rng.normal(1000, 500, size=(int(p.sum()), len(SENSORS)))
    fin = finalize_features(build_raw_features(tl, er, mt), ERROR_SENTINEL_HOURS)
    new = fin.loc[fin["datetime"] == t, ALL_FEATURES].to_numpy(dtype="float64")[0]
    return new, row[ALL_FEATURES].to_numpy(dtype="float64")

n_tests = n_bad = 0
for split in SPLIT_NAMES:
    for i in rng.choice(len(SPLIT_DF[split]), size=min(N_PER_SPLIT, len(SPLIT_DF[split])), replace=False):
        new, stored = recompute_row(split, int(i), "future")
        n_tests += 1
        n_bad += int(not np.allclose(new, stored, rtol=1e-5, atol=1e-5))
record("F-1 future perturbation leaves every tested feature vector unchanged", n_bad == 0,
       f"{n_tests} rows tested ({N_PER_SPLIT} per split), {n_bad} changed")

# positive control: perturbing the PAST must change the features (rows with a full 24 h history)
changed = tested = 0
tr_ok = np.flatnonzero(SPLIT_DF["train"]["past_observation_count_24h"].to_numpy() >= 24)
for i in rng.choice(tr_ok, size=N_CONTROL, replace=False):
    new, stored = recompute_row("train", int(i), "past")
    tested += 1
    changed += int(not np.allclose(new, stored, rtol=1e-5, atol=1e-5))
record("F-2 positive control: perturbing the past does change the features", changed == tested, f"{changed}/{tested} rows changed")

# --- F2/F3: scaler statistics are verified against the exact allowed source rows;
# held-out calibration is feature-only and disjoint from site evaluation.
saved = json.loads((CFG_DIR / "scaler_statistics.json").read_text())
disk = {n: pd.read_parquet(PROC_DIR / f"{n}.parquet") for n in ("train", "test_site_full")}
calibration_disk = pd.read_parquet(PROC_DIR / "heldout_calibration_features.parquet")

def source_rows(name):
    if name == "global":
        return disk["train"]
    if name == f"local_{HELD_OUT}":
        return calibration_disk
    return disk["train"][disk["train"]["client_id"] == name.replace("local_", "")]

calib_eval_keys = calibration_disk[["machineID", "datetime"]].merge(
    disk["test_site_full"][["machineID", "datetime"]], on=["machineID", "datetime"], how="inner")
calibration_valid = (
    TARGET not in calibration_disk.columns
    and list(calibration_disk.columns) == META_COLS + ALL_FEATURES
    and len(calib_eval_keys) == 0
    and calibration_disk["datetime"].max() < disk["test_site_full"]["datetime"].min()
)
ok = bool(calibration_valid)
for name, st in saved.items():
    X = source_rows(name)[ALL_FEATURES].to_numpy(dtype="float64")
    obj = joblib.load(SCALER_DIR / f"{name}.joblib")
    ok &= np.allclose(X.mean(0), st["mean"], rtol=1e-9, atol=1e-9)
    ok &= np.allclose(X.std(0), st["scale"], rtol=1e-9, atol=1e-9)
    ok &= np.allclose(obj.mean_, st["mean"]) and np.allclose(obj.scale_, st["scale"])
record("F-3 scaler stats match permitted source rows; held-out calibration is label-free and disjoint", bool(ok),
       f"{len(saved)} scalers checked; calibration rows={len(calibration_disk):,}; overlap rows={len(calib_eval_keys)}")

# --- F3: schema and feature/identifier separation
cols_ref = list(pd.read_parquet(PROC_DIR / "train.parquet").columns)
same_cols = all(list(pd.read_parquet(PROC_DIR / f"{n}.parquet").columns) == cols_ref for n in SPLIT_NAMES)
cfg = json.loads((CFG_DIR / "feature_groups.json").read_text())
record("F-4 identical schema in every split; features match feature_groups.json; no id/target leaks into features",
       same_cols and cfg["feature_sets"]["full"] == ALL_FEATURES
       and set(ALL_FEATURES).isdisjoint(META_COLS + [TARGET, "has_prior_maintenance"])
       and cols_ref == META_COLS + [TARGET] + ALL_FEATURES,
       f"{len(ALL_FEATURES)} features; no_maintenance set has {len(cfg['feature_sets']['no_maintenance'])}")

## 12. Final report

Writes `reports/data_prep_final_report.json` and `.md` with the PASS/FAIL checklist, split sizes, event counts, manifest details, and known limitations. A failed check stops the run. Because the split/partition protocol changed in v3, the previous v2 execution outputs are intentionally not carried over; run this notebook from top to bottom in the intended environment.


In [ ]:
# 12.1  Final report
import datetime as _dt

LIMITATIONS = [
    "The factories are simulated clients, not real factory records; this preprocessing notebook centralizes the raw tables to simulate the partition.",
    "The primary partition uses mixed machine types with different quotas, reducing the old single-model held-out confounding but not proving generalization to a real site. A model-blocked stress-test manifest is saved separately and must be evaluated in a separate experiment.",
    "The first 50% of the held-out timeline is used only for unsupervised local-scaler calibration. The feature-only artifact omits labels, and all site-evaluation rows are later than the calibration cutoff.",
    "test_site_late is a subset of test_site_full; these results are nested, not independent samples. Confidence intervals should be computed by machine or distinct failure event, not by treating hourly rows as independent.",
    "The data cover one year and each temporal split has a limited number of distinct failure events; per-factory and late-window metrics can be statistically fragile.",
    "Positive rows are not independent: one failure can produce up to 24 consecutive positive rows. The effective sample size is closer to the number of distinct failure events.",
    "Maintenance recency partly encodes time since the last failure because replacements may coincide with failures. It is past-only, included in 'full', and should be ablated using 'no_maintenance'.",
    "Rolling windows are row-count based and rely on the verified regular hourly grid (DQ-2).",
    "A global scaler is a centralized reference. The local scaler is an unsupervised adaptation policy; neither alone implements secure federated learning or a formal privacy guarantee.",
]

PROBLEM_TABLE = [
    ("Raw-data reproducibility", "Compare SHA-256, byte size and row count with an immutable prior manifest; fail on mismatch", "DATA-1, DATA-2"),
    ("Held-out calibration overlaps old site test", "Use a feature-only calibration artifact and start site evaluation after the calibration cutoff", "S-5, N-0, F-3"),
    ("Site/type confounding", "Use mixed-model quotas for primary clients and save a separate model-blocked stress-test manifest", "C-0, C-1, factory_partition_config.json"),
    ("Label construction", "Recompute the 24-hour label independently and test boundary handling", "L-1, L-2"),
    ("Temporal leakage", "Use 24-hour purge gaps, past-only features, future perturbation and a past-perturbation positive control", "S-1, F-1, F-2"),
    ("Feature/schema integrity", "Validate finite values, scaler source statistics, consistent schemas and exclusion of labels/IDs from features", "FE-1, F-3, F-4"),
    ("Machine-type stress generalization", "Save a model-blocked alternate manifest; it must be run as a separate evaluation scenario", "factory_manifest_type_shift_stress.csv"),
]

report = {
    "generated_utc": _dt.datetime.now(_dt.timezone.utc).isoformat(timespec="seconds"),
    "notebook_version": "prep_v3_revised",
    "environment": {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__},
    "checks": CHECKS,
    "all_checks_pass": all(c["status"] == "PASS" for c in CHECKS.values()),
    "boundaries": FEATURE_CONFIG["boundaries"],
    "factory_partition": {
        "primary_manifest": "manifests/factory_manifest.csv",
        "alternate_stress_manifest": "manifests/factory_manifest_type_shift_stress.csv",
        "model_quotas": MODEL_QUOTAS,
        "composition": composition.reset_index().to_dict(orient="records"),
        "legacy_comparison": legacy_comparison,
    },
    "normalization": {
        "policies": ["global", "local", "none"],
        "held_out_calibration_cutoff": str(CALIBRATION_CUTOFF),
        "held_out_calibration_fraction": HELD_OUT_CALIBRATION_FRACTION,
        "calibration_file": "processed/heldout_calibration_features.parquet",
        "calibration_contains_target": False,
        "evaluation_disjoint_from_calibration": True,
    },
    "splits": {n: {"rows": len(d), "machines": int(d["machineID"].nunique()),
                   "positive_rows": int(d[TARGET].sum()),
                   "positive_rate_pct": round(100 * d[TARGET].mean(), 4),
                   "factories": sorted(d["client_id"].unique())}
               for n, d in SPLIT_DF.items()},
    "held_out_calibration": {"rows": int(len(calibration_df)), "columns": list(calibration_df.columns),
                              "contains_target": TARGET in calibration_df.columns},
    "failure_events": EVENT_TABLE[EVENT_TABLE["factory"] == "ALL"].set_index("split")["failure_events"].to_dict(),
    "features": {"n_features": len(ALL_FEATURES), "groups": {g: len(c) for g, c in FEATURE_GROUPS.items()},
                 "primary_feature_set": "full", "ablation_feature_set": "no_maintenance",
                 "error_recency_sentinel_hours": ERROR_SENTINEL_HOURS},
    "maintenance_semantics": MAINT_SEMANTICS,
    "raw_data_checksums": checksums,
    "known_limitations": LIMITATIONS,
    "problem_to_change_to_evidence": [{"problem": a, "change": b, "evidence": c} for a, b, c in PROBLEM_TABLE],
}
(REP_DIR / "data_prep_final_report.json").write_text(json.dumps(report, indent=2, default=str))

md = ["# FedKG-Maint — Data preparation v3 final report", "", f"Generated (UTC): {report['generated_utc']}", "",
      "## Checklist", "", "| Check | Status | Detail |", "|---|---|---|"]
md += [f"| {k} | {v['status']} | {v['detail'][:140]} |" for k, v in CHECKS.items()]
md += ["", "## Final evaluation splits", "", "| Split | Rows | Machines | Positive rows | Positive rate % | Failure events |", "|---|---:|---:|---:|---:|---:|"]
md += [f"| {n} | {s['rows']:,} | {s['machines']} | {s['positive_rows']:,} | {s['positive_rate_pct']} | {report['failure_events'][n]} |"
       for n, s in report["splits"].items()]
md += ["", "## Held-out calibration", "",
       f"- Calibration cutoff: `{CALIBRATION_CUTOFF}`",
       f"- Rows: {len(calibration_df):,}",
       f"- Target column present: `{TARGET in calibration_df.columns}`",
       "- The held-out local scaler uses this feature-only file; evaluation rows begin at/after the cutoff.",
       "", "## Problem → change → evidence", "", "| Problem | Change | Evidence |", "|---|---|---|"]
md += [f"| {a} | {b} | {c} |" for a, b, c in PROBLEM_TABLE]
md += ["", "## Known limitations", ""] + [f"{i}. {l}" for i, l in enumerate(LIMITATIONS, 1)]
(REP_DIR / "data_prep_final_report.md").write_text("\n".join(md))

print(pd.DataFrame([{"check": k, "status": v["status"]} for k, v in CHECKS.items()]).to_string(index=False))
print("\nFinal evaluation splits:")
print(pd.DataFrame(report["splits"]).T[["rows", "machines", "positive_rows", "positive_rate_pct"]].assign(
    failure_events=pd.Series(report["failure_events"])).to_string())
print("\nHeld-out calibration:", len(calibration_df), "rows; target column absent:", TARGET not in calibration_df.columns)
print("\nReport saved to", REP_DIR / "data_prep_final_report.md")
assert report["all_checks_pass"], "At least one check failed"
print("\nDATA PREPARATION v3 COMPLETE — all checks PASS.")

## Phase 1 extension — Immutable run identity and non-IID diagnostics

A signed-by-hash **data protocol record** is frozen; score tables cannot use prior audit counts. Model proportions and event rates reveal how non-IID the clients are. The alternative model-blocked scenario is **a manifest only** and must not be mistaken for a separately trained stress result.

In [ ]:
import hashlib, math
manifest_path = MAN_DIR / 'factory_manifest.csv'
split_artifacts = [PROC_DIR / (s + '.parquet') for s in SPLIT_NAMES]
model_dist = pd.crosstab(manifest.client_id, manifest.model, normalize='index')
client_stats = train_df.groupby('client_id')[TARGET].agg(['size','sum','mean']).rename(columns={'size':'train_rows','sum':'positive_rows','mean':'positive_prevalence'})
client_stats = client_stats.join(model_dist.add_prefix('model_mix_'))
client_stats.to_csv(REP_DIR / 'client_non_iid_diagnostics.csv')
DATA_PROTOCOL = {
    'run_id':RUN_ID, 'raw_sha256':{k:v['sha256'] for k,v in checksums.items()},
    'factory_manifest_sha256':sha256_of(manifest_path),
    'split_artifact_sha256':{p.name:sha256_of(p) for p in split_artifacts},
    'factory_manifest_type':'mixed-model non-IID with 5 clients',
    'feature_config_sha256':sha256_of(CFG_DIR/'feature_groups.json'),
    'primary_training_factories':TRAINING_FACTORIES, 'held_out_factory':HELD_OUT,
    'target':TARGET, 'test_site_full_n':len(SPLIT_DF['test_site_full']),
    'calibration_is_feature_only':True,
    'primary_normalization':MODEL_POLICY,
    'note':'model-blocked alternative is not evaluated in this run',
}
(PROJECT_DIR/'data_protocol_lock.json').write_text(json.dumps(DATA_PROTOCOL, indent=2))
print(client_stats.round(4))
assert all((PROC_DIR / (s+'.parquet')).exists() for s in SPLIT_NAMES)

# Phase 2 — Validation-only centralized + Local-Only references

No held-out scores or labels are used for model hyperparameter selection in this section. Dataset preparation necessarily processes the raw events and held-out feature schema earlier. We train a small matched-architecture reference and full-feature logistic/HGB baselines. Per-factory local-only models are genuine **separate training runs**, evaluated locally on their own development-validation rows and later on the unseen site; the aggregated local ensemble is explicitly identified as a separate reference rather than a new FL algorithm.

In [ ]:
import importlib, joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, precision_score, recall_score
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore',category=ConvergenceWarning)
P2_DIR = PROJECT_DIR/'phase2_baselines_prep_v3'
for sub in ['models','reports','results','predictions']: (P2_DIR/sub).mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(PREP_DIR))
sys.modules.pop('fedkg_data',None); importlib.invalidate_caches()
import fedkg_data
P2_CONFIG = {'SEED':42,'BOOTSTRAP_B':BOOTSTRAP_B, 'CI_LEVEL':0.95,'PRIMARY_POLICY':'local',
             'HGB_GRID':[dict(max_iter=100,max_depth=3,learning_rate=.1),dict(max_iter=200,max_depth=6,learning_rate=.05)],
             'LR_C':[.05,.5,5.0]}
CONFIG = P2_CONFIG
fail_events = failures[['machineID','datetime']].drop_duplicates().sort_values(['machineID','datetime'])

In [ ]:
# 2.1  Write the evaluation module, import it, and verify it against scikit-learn
MODULE_SRC = '''"""Evaluation toolkit for FedKG-Maint (machine-level weighted metrics + cluster bootstrap).

    import sys; sys.path.insert(0, "<PROJECT_DIR>/phase2_baselines_prep_v3")
    from fedkg_eval import SetEval, draw_weights
    se = SetEval(meta, y, fail_events)          # meta: DataFrame with machineID, datetime sorted by (machine, time)
    pk = se.pack(scores); thr = se.best_f1_threshold(pk)       # threshold: fit on VALIDATION only
    point = se.metrics(pk, None, thr)
    reps = se.bootstrap(pk, thr, draw_weights(se.M, 1000, seed))
"""
import numpy as np
import pandas as pd

METRIC_COLS = ["ap", "roc_auc", "precision", "recall", "f1", "fpr", "event_recall"]


def draw_weights(n_machines, B, seed):
    """B bootstrap resamples of machines: integer counts per machine (rows sum to n_machines)."""
    rng = np.random.default_rng(seed)
    return rng.multinomial(n_machines, np.full(n_machines, 1.0 / n_machines), size=B).astype("float64")


def ci(values, level=0.95):
    a = (1.0 - level) / 2.0
    return tuple(np.percentile(np.asarray(values, dtype="float64"), [100 * a, 100 * (1 - a)]))


class SetEval:
    def __init__(self, meta, y, fail_events, horizon_hours=24):
        self.y = np.asarray(y, dtype="float64")
        self.n = len(self.y)
        mid = meta["machineID"].to_numpy()
        self.machines, self.mach_idx = np.unique(mid, return_inverse=True)
        self.mach_idx = self.mach_idx.ravel()
        self.M = len(self.machines)
        ts = meta["datetime"].to_numpy()
        assert np.all(np.diff(self.mach_idx) >= 0), "rows must be sorted by machine"
        assert np.all((np.diff(ts.astype("int64")) >= 0) | (np.diff(self.mach_idx) > 0)), "rows must be sorted by time inside a machine"
        starts = np.searchsorted(self.mach_idx, np.arange(self.M), side="left")
        ends = np.searchsorted(self.mach_idx, np.arange(self.M), side="right")
        H = np.timedelta64(int(horizon_hours), "h")
        slices, ev_m = [], []
        fe = fail_events[fail_events["machineID"].isin(self.machines)]
        for m, g in fe.groupby("machineID"):
            k = int(np.searchsorted(self.machines, m))
            s, e = int(starts[k]), int(ends[k])
            t_m = ts[s:e]
            f = g["datetime"].to_numpy()
            lo = np.searchsorted(t_m, f - H, side="left")      # rows with t >= f - 24h
            hi = np.searchsorted(t_m, f, side="left")          # rows with t <  f
            for a, b in zip(lo, hi):
                if b > a:
                    slices.append((s + int(a), s + int(b)))
                    ev_m.append(k)
        self.ev_slices = slices
        self.ev_mach = np.array(ev_m, dtype=int)
        self.n_events = len(slices)

    def pack(self, score):
        score = np.asarray(score, dtype="float64")
        if not np.isfinite(score).all():
            raise ValueError("non-finite scores")
        uniq, inv = np.unique(-score, return_inverse=True)     # group 0 = highest score; ties share a group
        return {"score": score, "inv": np.asarray(inv).ravel().astype(np.int64), "G": len(uniq), "thr_vals": -uniq}

    def best_f1_threshold(self, pk):
        tp_g = np.bincount(pk["inv"], self.y, pk["G"])
        fp_g = np.bincount(pk["inv"], 1.0 - self.y, pk["G"])
        tp, fp = np.cumsum(tp_g), np.cumsum(fp_g)
        f1 = 2.0 * tp / (tp + fp + tp[-1])
        return float(pk["thr_vals"][int(np.argmax(f1))])

    def _event_flags(self, pk, thr):
        if self.n_events == 0:
            return np.zeros(0, dtype=bool)
        if "ev_max" not in pk:
            pk["ev_max"] = np.array([pk["score"][a:b].max() for a, b in self.ev_slices])
        return pk["ev_max"] >= thr

    def metrics(self, pk, wm=None, thr=None):
        wr = np.ones(self.n) if wm is None else wm[self.mach_idx]
        wy = wr * self.y
        wn = wr - wy
        tp_g = np.bincount(pk["inv"], wy, pk["G"])
        fp_g = np.bincount(pk["inv"], wn, pk["G"])
        tp, fp = np.cumsum(tp_g), np.cumsum(fp_g)
        P, N = float(tp[-1]), float(fp[-1])
        out = {"prevalence": P / (P + N) if P + N > 0 else np.nan}
        if P <= 0 or N <= 0:
            out.update({k: np.nan for k in METRIC_COLS})
            return out
        denom = tp + fp
        prec = np.divide(tp, denom, out=np.zeros_like(tp), where=denom > 0)
        rec = tp / P
        out["ap"] = float(np.sum(np.diff(np.concatenate(([0.0], rec))) * prec))
        out["roc_auc"] = float(np.sum(tp_g * (N - fp + 0.5 * fp_g)) / (P * N))
        if thr is not None:
            pred = pk["score"] >= thr
            tpt, fpt = float(np.sum(wy * pred)), float(np.sum(wn * pred))
            p_ = tpt / (tpt + fpt) if tpt + fpt > 0 else 0.0
            r_ = tpt / P
            out.update({"precision": p_, "recall": r_, "f1": 2 * p_ * r_ / (p_ + r_) if p_ + r_ > 0 else 0.0, "fpr": fpt / N})
            wev = np.ones(self.n_events) if wm is None else wm[self.ev_mach]
            flag = self._event_flags(pk, thr)
            out["event_recall"] = float(np.sum(wev * flag) / np.sum(wev)) if np.sum(wev) > 0 else np.nan
        else:
            # A constant prevalence score has no meaningful operating threshold.
            out.update({k: np.nan for k in ["precision", "recall", "f1", "fpr", "event_recall"]})
        return out

    def bootstrap(self, pk, thr, W):
        return pd.DataFrame([self.metrics(pk, w, thr) for w in W])
'''
(P2_DIR / "fedkg_eval.py").write_text(MODULE_SRC)
sys.path.insert(0, str(P2_DIR))
sys.modules.pop("fedkg_eval", None)  # ensure the module just written to this versioned output directory is used
importlib.invalidate_caches()
import fedkg_eval
from fedkg_eval import SetEval, draw_weights, ci

# ---- verification against scikit-learn (with tied scores) and against physically replicated rows
rng = np.random.default_rng(CONFIG["SEED"])
n_m, per = 20, 200
meta_t = pd.DataFrame({"machineID": np.repeat(np.arange(1, n_m + 1), per),
                       "datetime": np.tile(pd.date_range("2015-01-01", periods=per, freq="h").to_numpy(), n_m)})
y_t = (rng.random(n_m * per) < 0.08).astype(float)
s_t = np.round(1.5 * y_t + rng.normal(0, 1, n_m * per), 1)                       # many ties on purpose
no_events = pd.DataFrame({"machineID": pd.Series(dtype="int64"), "datetime": pd.Series(dtype="datetime64[ns]")})
se_t = SetEval(meta_t, y_t, no_events)
m0 = se_t.metrics(se_t.pack(s_t))
ok_sk = abs(m0["ap"] - average_precision_score(y_t, s_t)) < 1e-9 and abs(m0["roc_auc"] - roc_auc_score(y_t, s_t)) < 1e-9
counts = rng.multinomial(n_m, np.full(n_m, 1 / n_m))
rows_rep = np.repeat(np.arange(len(y_t)), counts[se_t.mach_idx].astype(int))
mw = se_t.metrics(se_t.pack(s_t), counts.astype(float))
ok_rep = (abs(mw["ap"] - average_precision_score(y_t[rows_rep], s_t[rows_rep])) < 1e-9
          and abs(mw["roc_auc"] - roc_auc_score(y_t[rows_rep], s_t[rows_rep])) < 1e-9)
record("E-1 weighted AP / ROC-AUC equal scikit-learn (ties) and equal metrics on physically replicated rows", bool(ok_sk and ok_rep),
       f"AP={m0['ap']:.6f} vs sklearn {average_precision_score(y_t, s_t):.6f}")

# event window: failure at hour 50 -> rows t with t < f <= t + 24h  =>  rows 26..49
meta_e = meta_t[meta_t["machineID"] == 1].reset_index(drop=True)
fe_t = pd.DataFrame({"machineID": [1], "datetime": [meta_e["datetime"].iloc[50]]})
se_e = SetEval(meta_e, np.zeros(len(meta_e)), fe_t)
record("E-2 event window = the 24 rows strictly before the failure", se_e.ev_slices == [(26, 50)], f"slice={se_e.ev_slices}")

zero_w = se_t.metrics(se_t.pack(s_t), np.r_[np.zeros(n_m - 1), n_m].astype(float))   # one machine only: must not crash
print("Toolkit written to", P2_DIR / "fedkg_eval.py")

In [ ]:
from fedkg_eval import SetEval, draw_weights, ci
Xv_full, yv, mv = fedkg_data.load_split('validation',policy='local',feature_set='full')
VAL_EVAL = SetEval(mv, yv, fail_events[fail_events.machineID.isin(mv.machineID.unique())])
model_candidates = []
central_models = {}; central_locks = []
feature_sets = ['full','no_maintenance']
for feature_set in feature_sets:
    for policy in ['local','global']:
        Xtr_b, ytr_b, mtr_b = fedkg_data.load_split('train',policy=policy,feature_set=feature_set)
        Xva_b, yva_b, mva_b = fedkg_data.load_split('validation',policy=policy,feature_set=feature_set)
        assert np.array_equal(yva_b,yv) and mva_b[['machineID','datetime','client_id']].equals(mv)
        for C in P2_CONFIG['LR_C']:
            model=LogisticRegression(C=C,solver='lbfgs',max_iter=400,class_weight='balanced')
            model.fit(Xtr_b,ytr_b)
            score=model.predict_proba(Xva_b)[:,1]
            model_candidates.append((float(average_precision_score(yv,score)), 'logreg', policy, feature_set, {'C':C},model))
        if ENABLE_HGB:
            for hp in P2_CONFIG['HGB_GRID']:
                model=HistGradientBoostingClassifier(random_state=42,**hp)
                model.fit(Xtr_b,ytr_b)
                score=model.predict_proba(Xva_b)[:,1]
                model_candidates.append((float(average_precision_score(yv,score)), 'hgb', policy,feature_set,hp,model))
        print('Completed validation candidates:',policy,feature_set)
# IMPORTANT: select by validation only, keeping interpretable local/full and maintenance ablations.
for family in ['logreg','hgb'] if ENABLE_HGB else ['logreg']:
    for policy in ['local','global']:
        for fs in feature_sets:
            choices=[r for r in model_candidates if r[1:4]==(family,policy,fs)]
            win=max(choices,key=lambda z:z[0]); ap,fam,pol,feature_set,hp,model=win
            key=f'{fam}__{pol}__{feature_set}'
            central_models[key]={'model':model,'policy':pol,'feature_set':feature_set,'val_ap':ap,'hparams':hp}
            joblib.dump(model,P2_DIR/'models'/f'{key}.joblib')
            central_locks.append({'model':key,'validation_ap':ap,'hparams':hp,'selection_split':'validation'})
# Local-only train EACH development factory independently: never tune on factory_5.
local_only={}
Xtr_l,ytr_l,mtr_l=fedkg_data.load_split('train',policy='local',feature_set='full')
Xva_l,yva_l,mva_l=fedkg_data.load_split('validation',policy='local',feature_set='full')
for f in TRAINING_FACTORIES:
    im=(mtr_l.client_id==f).to_numpy(); vm=(mva_l.client_id==f).to_numpy()
    choices=[]
    for C in P2_CONFIG['LR_C']:
        m=LogisticRegression(C=C,solver='lbfgs',max_iter=400,class_weight='balanced')
        m.fit(Xtr_l[im],ytr_l[im]); scr=m.predict_proba(Xva_l[vm])[:,1]
        ap=float(average_precision_score(yva_l[vm],scr))
        choices.append((ap,C,m))
    ap,C,best=max(choices,key=lambda z:z[0]); local_only[f]=best
    joblib.dump(best,P2_DIR/'models'/f'localonly__{f}.joblib')
    central_locks.append({'model':f'localonly__{f}','validation_ap':ap,'hparams':{'C':C},'selection_split':f'validation__{f}'})
assert not any('test_site' in str(v) for v in central_locks)
(P2_DIR/'reports'/'validation_only_selection_lock.json').write_text(json.dumps(central_locks,indent=2))
pd.DataFrame(central_locks).to_csv(P2_DIR/'results'/'validation_selection.csv',index=False)
print(pd.DataFrame(central_locks)[['model','validation_ap']].to_string(index=False))

# Phase 3 — Real FL algorithms + scoped Differential Privacy

**Training choices frozen before test access:** 7 compact features (four current sensors, error recency and indicator, maintenance recency), `ROUNDS`, one local epoch, learning rate, clipping, noise, FedProx μ. Seeds 42–44. No search over DP settings is performed (avoids privacy-accounting ambiguity from repeated tuning). FedAvg and FedProx genuinely differ in their training objective. DP-FedAvg uses clipped individual client updates and Gaussian noise on the released aggregate; the client data remain in one simulation process.

All algorithms use **local per-client normalization**; Global-scaler comparisons are centralized-only references. The privacy bound is an **add/remove-one-client** bound for the released Gaussian mechanism with a trusted server, fixed denominator and full participation. Per-round validation metrics and LLM/graph releases are not within that bound.

In [ ]:
import torch
from torch import nn
from torch.nn.utils import parameters_to_vector, vector_to_parameters
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(min(4,max(1,os.cpu_count() or 1)))
CLIENTS=list(TRAINING_FACTORIES)
FEATURE_CFG=json.loads((PREP_DIR/'configs'/'feature_groups.json').read_text())
FEATURE_NAMES=(FEATURE_CFG['groups']['sensor_raw'] + FEATURE_CFG['groups']['error_recency'] + FEATURE_CFG['groups']['maintenance_recency'])
FEATURE_IDXS=[FEATURE_CFG['feature_sets']['full'].index(n) for n in FEATURE_NAMES]
XtrA,ytrA,mtrA=fedkg_data.load_split('train',policy=MODEL_POLICY,feature_set='full')
XvaA,yvaA,mvaA=fedkg_data.load_split('validation',policy=MODEL_POLICY,feature_set='full')
assert MODEL_POLICY=='local' and np.array_equal(yvaA,yv)
assert HELD_OUT not in set(mtrA.client_id) and HELD_OUT not in set(mvaA.client_id)
Xtr=XtrA[:,FEATURE_IDXS].copy();Xv=XvaA[:,FEATURE_IDXS].copy()
CLIENT_ROW_IDX={f:np.flatnonzero(mtrA.client_id.to_numpy()==f) for f in CLIENTS}
assert all(len(v)>0 for v in CLIENT_ROW_IDX.values())
assert len(set(len(v) for v in CLIENT_ROW_IDX.values()))==1,'DP adjacency proof uses equal clients + fixed denominator'
P3_DIR=PROJECT_DIR/'phase3_federated_localnorm'
for sub in ['models','reports','results','predictions','logs']:(P3_DIR/sub).mkdir(parents=True,exist_ok=True)
FL_PARAMETERS={
    'model':'single linear layer / 7 features / 8 parameters','methods':['FedAvg','FedProx','ClippedFedAvg','DP-FedAvg','CentralCompactLinear'],
    'clients':CLIENTS,'site_held_out':HELD_OUT,'rounds':ROUNDS,'local_epochs':LOCAL_EPOCHS,
    'batch_size':4096,'learning_rate':1e-3,'weight_decay':1e-5,'pos_weight':20.0,
    'clip_norm':CLIP_NORM,'noise_multiplier':NOISE_MULTIPLIER,'delta':DP_DELTA,
    'num_clients_ablation':[2,3,4],
    'fedprox_mu':0.5,'seeds':SEEDS,'normalization_policy':MODEL_POLICY,'features':FEATURE_NAMES
}
(P3_DIR/'reports'/'frozen_training_protocol.json').write_text(json.dumps(FL_PARAMETERS,indent=2))
print('Model features',FEATURE_NAMES,'train',Xtr.shape,'val',Xv.shape,'device',DEVICE)

In [ ]:
# Predeclared RDP bound and independent accountant cross-check.
import math
ORDERS=np.asarray([1.25,1.5,2,3,4,5,8,10,16,32,64,128,256,512],dtype=float)
def conservative_rdp_epsilon(sigma, rounds=ROUNDS, delta=DP_DELTA):
    if sigma<=0: return float('inf')
    return float(np.min(rounds*ORDERS/(2*sigma**2)+np.log(1/delta)/(ORDERS-1)))
EPSILON_MANUAL=conservative_rdp_epsilon(NOISE_MULTIPLIER)
from dp_accounting import dp_event, rdp
accountant=rdp.RdpAccountant(orders=ORDERS.tolist())
accountant.compose(dp_event.GaussianDpEvent(NOISE_MULTIPLIER),count=ROUNDS)
EPSILON_LIBRARY=float(accountant.get_epsilon(DP_DELTA))
record('DP-1 independent RDP accountants meet predeclared epsilon cap',
       np.isfinite(EPSILON_MANUAL) and EPSILON_MANUAL<=EPSILON_CAP and EPSILON_LIBRARY<=EPSILON_CAP,
       f'manual={EPSILON_MANUAL:.4f}; dp-accounting={EPSILON_LIBRARY:.4f}; delta={DP_DELTA}')
PRIVACY_SCOPE={'epsilon_manual_conservative':EPSILON_MANUAL,'epsilon_library':EPSILON_LIBRARY,'delta':DP_DELTA,
    'covered_release':'three or configured number of noisy clipped aggregate updates; add/remove one factory',
    'trusted_server':True,'fixed_denominator':len(CLIENTS),'full_participation':True,'subsampling_amplification':False,
    'NOT_COVERED':['raw-data preprocessing','local normalization parameters','validation selection and metrics',
                   'local-model outputs','non-DP baselines','knowledge graphs','evaluation raw labels','test diagnostics'],
    'noise_rng':'OS-entropy-backed SystemRandom Gaussian draws; training seeds do not reveal DP noise',
    'interpretation':'mechanism-scoped client-level DP, not full-system DP; centralized simulation only'}
(P3_DIR/'reports'/'privacy_scope.json').write_text(json.dumps(PRIVACY_SCOPE,indent=2))
print(json.dumps(PRIVACY_SCOPE,indent=2))

In [ ]:
# Fixed-seed, fixed-round reference implementation. Exactly one client delta is released per client per round.
import secrets
SECURE_DP_RNG=secrets.SystemRandom()
BATCH_SIZE=FL_PARAMETERS['batch_size'];LR=FL_PARAMETERS['learning_rate'];WD=FL_PARAMETERS['weight_decay'];POS_WEIGHT=FL_PARAMETERS['pos_weight']
T_X=torch.as_tensor(Xtr,dtype=torch.float32,device=DEVICE)
T_Y=torch.as_tensor(ytrA.reshape(-1,1),dtype=torch.float32,device=DEVICE)
CLIENT_T={f:torch.as_tensor(idx,dtype=torch.long,device=DEVICE) for f,idx in CLIENT_ROW_IDX.items()}

def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
def new_linear(n_features=7):
    mod=nn.Linear(n_features,1).to(DEVICE)
    with torch.no_grad():mod.weight.zero_();mod.bias.zero_()
    return mod
@torch.no_grad()
def vector_of(m):return parameters_to_vector(m.parameters()).detach().clone()
def replace_vector(m,v):vector_to_parameters(v,m.parameters())
@torch.no_grad()
def linear_predict(m,X):
    m.eval();parts=[]
    for st in range(0,len(X),32768):
        tx=torch.as_tensor(X[st:st+32768],dtype=torch.float32,device=DEVICE)
        parts.append(torch.sigmoid(m(tx).reshape(-1)).cpu().numpy())
    return np.concatenate(parts)
def do_client_epochs(local,rows,reference=None,mu=0.0,epochs=1):
    opt=torch.optim.Adam(local.parameters(),lr=LR,weight_decay=WD)
    crit=nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT],device=DEVICE))
    local.train()
    for ep in range(epochs):
        idx=rows[torch.randperm(len(rows),device=DEVICE)]
        for st in range(0,len(idx),BATCH_SIZE):
            ii=idx[st:st+BATCH_SIZE]
            opt.zero_grad(set_to_none=True)
            loss=crit(local(T_X[ii]),T_Y[ii])
            if reference is not None and mu>0:
                # FedProx: (mu/2) ||w_local - w_global||^2, inside the client loss.
                loss=loss+(mu/2.0)*torch.sum((parameters_to_vector(local.parameters())-reference)**2)
            loss.backward();opt.step()

def fl_train(seed,method,participating=None):
    assert method in ['FedAvg','FedProx','ClippedFedAvg','DP-FedAvg']
    participating=list(CLIENTS if participating is None else participating)
    assert len(participating)>=2 and set(participating)<=set(CLIENTS)
    seed_all(seed);global_model=new_linear(Xtr.shape[1]);logs=[]
    for rnd in range(1,ROUNDS+1):
        before=vector_of(global_model);deltas=[];uncapped=[]
        for f in participating:
            local_model=new_linear(Xtr.shape[1]);replace_vector(local_model,before.clone())
            do_client_epochs(local_model,CLIENT_T[f],reference=before if method=='FedProx' else None,
                             mu=FL_PARAMETERS['fedprox_mu'] if method=='FedProx' else 0.0,
                             epochs=LOCAL_EPOCHS)
            delta=vector_of(local_model)-before
            n=float(torch.linalg.vector_norm(delta).cpu());uncapped.append(n)
            if method in ('ClippedFedAvg','DP-FedAvg'):
                delta=delta*min(1.,CLIP_NORM/(n+1e-12))
            deltas.append(delta)
            del local_model
        aggregate=torch.stack(deltas).mean(0)
        if method=='DP-FedAvg':
            # Draw update noise from OS-backed entropy; published experiment seeds do NOT reveal the noise draw.
            dp_sd=NOISE_MULTIPLIER*CLIP_NORM/len(participating)
            noise=torch.as_tensor([SECURE_DP_RNG.gauss(0.0,dp_sd) for _ in range(aggregate.numel())],
                                  dtype=aggregate.dtype,device=aggregate.device).reshape(aggregate.shape)
            aggregate=aggregate+noise
        replace_vector(global_model,before+aggregate)
        logs.append({'method':method,'seed':seed,'round':rnd,'unclipped_max_norm':max(uncapped),
                     'unclipped_mean_norm':float(np.mean(uncapped)),
                     'fraction_clipped':float(np.mean(np.asarray(uncapped)>CLIP_NORM)) if 'Clipped' in method or method=='DP-FedAvg' else np.nan,
                     'validation_ap_diagnostic':float(average_precision_score(yvaA,linear_predict(global_model,Xv)))})
    val_score=linear_predict(global_model,Xv)
    thr=float(VAL_EVAL.best_f1_threshold(VAL_EVAL.pack(val_score)))
    state={k:v.detach().cpu().clone() for k,v in global_model.state_dict().items()}
    return {'method':method,'seed':seed,'state':state,'threshold':thr,
            'validation_ap':float(average_precision_score(yvaA,val_score)),'history':logs}

def central_compact_train(seed):
    seed_all(seed); m=new_linear(Xtr.shape[1]); idx=torch.arange(len(Xtr),device=DEVICE)
    # Same total epochs as FL rounds times client local epochs; central reference, not mathematically identical SGD budget.
    do_client_epochs(m,idx,epochs=ROUNDS*LOCAL_EPOCHS)
    scr=linear_predict(m,Xv)
    return {'method':'CentralCompactLinear','seed':seed,
            'state':{k:v.detach().cpu().clone() for k,v in m.state_dict().items()},
            'threshold':float(VAL_EVAL.best_f1_threshold(VAL_EVAL.pack(scr))),
            'validation_ap':float(average_precision_score(yvaA,scr)),'history':[]}

FL_ARTIFACTS={};FL_HISTORY=[]
for seed in SEEDS:
    for method in ['FedAvg','FedProx','ClippedFedAvg','DP-FedAvg','CentralCompactLinear']:
        r=central_compact_train(seed) if method=='CentralCompactLinear' else fl_train(seed,method)
        FL_ARTIFACTS[(method,seed)]={k:v for k,v in r.items() if k!='history'}
        FL_HISTORY.extend(r['history'])
        torch.save({'state':r['state'],'threshold':r['threshold'],'method':method,'seed':seed,
                    'validation_ap':r['validation_ap'],'feature_names':FEATURE_NAMES},
                   P3_DIR/'models'/f'{method}__seed{seed}.pt')
        print('Validation:',method,seed,'AP',round(r['validation_ap'],5))
pd.DataFrame(FL_HISTORY).to_csv(P3_DIR/'logs'/'client_norms_and_validation.csv',index=False)
val_summary=pd.DataFrame([{'method':m,'seed':s,'validation_ap':r['validation_ap'],'threshold':r['threshold']}
                          for (m,s),r in FL_ARTIFACTS.items()])
val_summary.to_csv(P3_DIR/'results'/'validation_only_fl_scores.csv',index=False)
# Number-of-client ablation: same fixed architecture, only participating factories change.
# This is a separate exploratory comparison, predeclared BEFORE the held-out evaluation.
NUM_CLIENT_ARTIFACTS={};NUM_CLIENT_VALIDATION=[]
for n_client in [2,3]:
    subset=CLIENTS[:n_client]
    for seed in SEEDS:
        r=fl_train(seed,'FedAvg',participating=subset)
        NUM_CLIENT_ARTIFACTS[(n_client,seed)]={'state':r['state'],'threshold':r['threshold'],'validation_ap':r['validation_ap']}
        NUM_CLIENT_VALIDATION.append({'n_clients':n_client,'seed':seed,'validation_ap':r['validation_ap']})
        torch.save({'state':r['state'],'threshold':r['threshold'],'n_clients':n_client,'seed':seed},
                   P3_DIR/'models'/f'FedAvg_clients{n_client}__seed{seed}.pt')
for seed in SEEDS:
    r=FL_ARTIFACTS[('FedAvg',seed)]
    NUM_CLIENT_ARTIFACTS[(4,seed)]=r
    NUM_CLIENT_VALIDATION.append({'n_clients':4,'seed':seed,'validation_ap':r['validation_ap']})
pd.DataFrame(NUM_CLIENT_VALIDATION).to_csv(P3_DIR/'results'/'client_number_ablation_validation.csv',index=False)

# selection-lock prevents interpreting reruns as blind after observing test.
LOCK={'run_id':RUN_ID,'test_performance_used_for_training_selection':False,
      'data_protocol_sha256':sha256_of(PROJECT_DIR/'data_protocol_lock.json'),
      'central_selection_sha256':sha256_of(P2_DIR/'reports'/'validation_only_selection_lock.json'),
      'fl_protocol':FL_PARAMETERS,'validation_scores':val_summary.to_dict('records'),
      'privacy':PRIVACY_SCOPE,'client_count_ablation_validation':NUM_CLIENT_VALIDATION,'model_choice':'all pre-registered methods and seeds are evaluated on held-out test',
      'no_hyperparameters_selected_using_test':True}
LOCK_PATH=PROJECT_DIR/'FINAL_SELECTION_LOCK.json'
if LOCK_PATH.exists():
    assert json.loads(LOCK_PATH.read_text())==LOCK,'Refusing to silently change an existing selection lock.'
else: LOCK_PATH.write_text(json.dumps(LOCK,indent=2,default=str))
print('Frozen final model-selection lock:',LOCK_PATH)

# Phase 3 closure — One post-lock held-out evaluation and machine-cluster uncertainty

**Do not rerun or modify training after seeing this section's test results.** The held-out features and labels were prepared and schema-checked in Phase 1; their **predictive performance is assessed only here**, after model-selection locks exist. Global-scaled centralized models are recorded as *centralized reference only*. Machine-cluster bootstrap uses machine as resampling unit; repeated hourly positives are not independent. Test-site-late is a nested sensitivity subset, not a separate independent test. The held-out factory was previously examined in the uploaded project history; this new run must not be described as a never-seen external benchmark.

In [ ]:
assert LOCK_PATH.exists() and sha256_of(PROJECT_DIR/'data_protocol_lock.json')==LOCK['data_protocol_sha256']
# Held-out predictive evaluation starts after selection lock. Split integrity was checked during preprocessing.
Xtest_l, ytest, mtest=fedkg_data.load_split('test_site_full',policy='local',feature_set='full')
Xt_compact=Xtest_l[:,FEATURE_IDXS]
TEST_EVAL=SetEval(mtest,ytest,fail_events[fail_events.machineID.isin(mtest.machineID.unique())])
assert set(mtest.client_id)=={HELD_OUT}
assert not mtest.duplicated(['machineID','datetime']).any()
assert len(mtest)==len(ytest) and np.isfinite(Xtest_l).all()
# Sanity: no target-factory rows in central/local-only training.
assert HELD_OUT not in set(mtrA.client_id)

def evaluate_saved_scores(name,seed,scores,val_scores,val_meta,threshold=None):
    assert len(scores)==len(ytest) and np.all(np.isfinite(scores))
    if threshold is None:
        ev=SetEval(val_meta,yv,fail_events[fail_events.machineID.isin(val_meta.machineID.unique())])
        threshold=float(ev.best_f1_threshold(ev.pack(val_scores)))
    pk=TEST_EVAL.pack(scores)
    mt=TEST_EVAL.metrics(pk,None,threshold)
    result={'model':name,'seed':seed,'threshold_from_validation':threshold,'test_rows':len(ytest),
            'machines':int(mtest.machineID.nunique()),'prevalence':float(ytest.mean()),**mt}
    return result,pk

test_rows=[];SAVED_SCORE_PACKS={};THRESHOLDS={}
for key,info in central_models.items():
    X_t,y_t,m_t=fedkg_data.load_split('test_site_full',policy=info['policy'],feature_set=info['feature_set'])
    X_v,y_v,m_v=fedkg_data.load_split('validation',policy=info['policy'],feature_set=info['feature_set'])
    assert np.array_equal(y_t,ytest) and m_t.equals(mtest) and np.array_equal(y_v,yv)
    ps=info['model'].predict_proba(X_t)[:,1];vs=info['model'].predict_proba(X_v)[:,1]
    res,pk=evaluate_saved_scores(key,42,ps,vs,m_v)
    test_rows.append(res);SAVED_SCORE_PACKS[(key,42)]=pk;THRESHOLDS[(key,42)]=res['threshold_from_validation']
    pd.DataFrame({'machineID':mtest.machineID,'datetime':mtest.datetime,'score':ps}).to_parquet(P2_DIR/'predictions'/f'{key}.parquet',index=False)
# Local-only models: test all four separately AND their simple ensemble, never call it FL.
local_ps=[];local_vs=[]
for f,model in local_only.items():
    ps=model.predict_proba(Xtest_l)[:,1]
    vs=model.predict_proba(Xva_l)[:,1]
    res,pk=evaluate_saved_scores('LocalOnly__'+f,42,ps,vs,mva_l)
    test_rows.append(res);SAVED_SCORE_PACKS[('LocalOnly__'+f,42)]=pk
    THRESHOLDS[('LocalOnly__'+f,42)]=res['threshold_from_validation']
    local_ps.append(ps);local_vs.append(vs)
ps=np.mean(local_ps,axis=0);vs=np.mean(local_vs,axis=0)
res,pk=evaluate_saved_scores('LocalOnly_Ensemble_Reference',42,ps,vs,mva_l)
test_rows.append(res);SAVED_SCORE_PACKS[('LocalOnly_Ensemble_Reference',42)]=pk
THRESHOLDS[('LocalOnly_Ensemble_Reference',42)]=res['threshold_from_validation']

for (method,seed),art in FL_ARTIFACTS.items():
    m=new_linear(len(FEATURE_NAMES));m.load_state_dict(art['state'])
    scores=linear_predict(m,Xt_compact)
    v_scores=linear_predict(m,Xv)
    res,pk=evaluate_saved_scores(method,seed,scores,v_scores,mvaA,threshold=art['threshold'])
    test_rows.append(res);SAVED_SCORE_PACKS[(method,seed)]=pk;THRESHOLDS[(method,seed)]=art['threshold']
    pd.DataFrame({'machineID':mtest.machineID,'datetime':mtest.datetime,'score':scores}).to_parquet(
        P3_DIR/'predictions'/f'{method}__seed{seed}.parquet',index=False)
metrics_df=pd.DataFrame(test_rows)
metrics_df.to_csv(P3_DIR/'results'/'heldout_model_comparison.csv',index=False)
print(metrics_df[['model','seed','ap','roc_auc','precision','recall','f1','event_recall']].sort_values('ap',ascending=False).round(4).to_string(index=False))

## Number of participating factories — held-out utility sensitivity

FedAvg with 2, 3 or 4 participating training factories; full training parameters and seeds are fixed before assessing the held-out site. Unlike model selection, this is a **post-lock descriptive ablation**; the experiments share the same test factory and cannot be interpreted as independent tests.

In [ ]:
N_CLIENT_ROWS=[]
for (n_client,seed),art in NUM_CLIENT_ARTIFACTS.items():
    model=new_linear(len(FEATURE_NAMES));model.load_state_dict(art['state'])
    scores=linear_predict(model,Xt_compact)
    mt=TEST_EVAL.metrics(TEST_EVAL.pack(scores),None,art['threshold'])
    N_CLIENT_ROWS.append({'clients':n_client,'seed':seed,'mean_communication_payload_bytes':
                          2*n_client*(len(FEATURE_NAMES)+1)*4*ROUNDS,'rows':len(ytest),'machines':TEST_EVAL.M,**mt})
client_count_df=pd.DataFrame(N_CLIENT_ROWS)
client_count_df.to_csv(P3_DIR/'results'/'participating_client_count_sensitivity.csv',index=False)
print(client_count_df.groupby('clients').agg(mean_pr_auc=('ap','mean'),std_pr_auc=('ap','std'),n_seeds=('seed','nunique')).round(5))

In [ ]:
# Fair paired machine bootstrap: same 20-machine resamples for all seeds and both methods.
W=draw_weights(TEST_EVAL.M,BOOTSTRAP_B,seed=62026)
paired=[];bootstrap_means=[]
for method in ['FedAvg','FedProx','ClippedFedAvg','DP-FedAvg','CentralCompactLinear']:
    for seed in SEEDS:
        pk=SAVED_SCORE_PACKS[(method,seed)]
        apps=np.asarray([TEST_EVAL.metrics(pk,w,None)['ap'] for w in W],dtype=float)
        bootstrap_means.append({'method':method,'seed':seed,'bootstrap_ap_mean':np.mean(apps),
                                'ci_low':np.quantile(apps,.025),'ci_high':np.quantile(apps,.975)})
for method_b in ['ClippedFedAvg','FedAvg','FedProx']:
    draws=[]
    for w in W:
        a=np.mean([TEST_EVAL.metrics(SAVED_SCORE_PACKS[('DP-FedAvg',s)],w,None)['ap'] for s in SEEDS])
        b=np.mean([TEST_EVAL.metrics(SAVED_SCORE_PACKS[(method_b,s)],w,None)['ap'] for s in SEEDS])
        draws.append(a-b)
    draws=np.asarray(draws)
    paired.append({'comparison':f'DP-FedAvg minus {method_b}',
        'mean_point_diff':float(metrics_df[metrics_df.model=='DP-FedAvg']['ap'].mean()-metrics_df[metrics_df.model==method_b]['ap'].mean()),
        'bootstrap_ci_low':float(np.quantile(draws,.025)),'bootstrap_ci_high':float(np.quantile(draws,.975)),
        'draw_fraction_leq_zero_descriptive':float(np.mean(draws<=0)),'B':BOOTSTRAP_B})
pd.DataFrame(bootstrap_means).to_csv(P3_DIR/'results'/'machine_bootstrap_by_seed.csv',index=False)
pd.DataFrame(paired).to_csv(P3_DIR/'results'/'paired_dp_comparisons.csv',index=False)
print(pd.DataFrame(paired).round(5).to_string(index=False))
# Model-update communication accounting (float32 model only, excluding headers, metadata, secure protocols or compression).
PARAM_COUNT=len(FEATURE_NAMES)+1
BYTES_ONE_UPDATE=4*PARAM_COUNT
COMM={'n_clients':len(CLIENTS),'rounds':ROUNDS,'trainable_parameters':PARAM_COUNT,
      'bytes_per_client_update_fp32':BYTES_ONE_UPDATE,
      'uplink_bytes_all_rounds':BYTES_ONE_UPDATE*len(CLIENTS)*ROUNDS,
      'downlink_bytes_all_rounds':BYTES_ONE_UPDATE*len(CLIENTS)*ROUNDS,
      'total_payload_bytes_excluding_protocol_overhead':2*BYTES_ONE_UPDATE*len(CLIENTS)*ROUNDS}
(P3_DIR/'reports'/'communication_cost_payload_only.json').write_text(json.dumps(COMM,indent=2))
print('Communication payload estimate:',COMM)

## Pre-registered sensitivity — late factory and late time tests

These are **nested/related** to other time windows and are never counted as independent samples. Use the previously trained artifacts and the same validation-derived thresholds. No new model search uses these sets.

In [ ]:
SECONDARY=[]
for split in ['test_site_late','test_time_late']:
    xs,ys,ms=fedkg_data.load_split(split,policy='local',feature_set='full')
    fe_sub=fail_events[fail_events.machineID.isin(ms.machineID.unique())]
    evaluator=SetEval(ms,ys,fe_sub)
    for method in ['FedAvg','FedProx','ClippedFedAvg','DP-FedAvg','CentralCompactLinear']:
        for seed in SEEDS:
            art=FL_ARTIFACTS[(method,seed)]
            model=new_linear(len(FEATURE_NAMES));model.load_state_dict(art['state'])
            pred=linear_predict(model,xs[:,FEATURE_IDXS])
            met=evaluator.metrics(evaluator.pack(pred),None,art['threshold'])
            SECONDARY.append({'split':split,'model':method,'seed':seed,'rows':len(ys),
                              'machines':ms.machineID.nunique(),'prevalence':float(ys.mean()),**met})
# Centralized reference uses its own correctly fitted scaler, for both time and held-out sites.
for split in ['test_site_late','test_time_late']:
    for key,info in central_models.items():
        xs,ys,ms=fedkg_data.load_split(split,policy=info['policy'],feature_set=info['feature_set'])
        test_e=SetEval(ms,ys,fail_events[fail_events.machineID.isin(ms.machineID.unique())])
        pv=info['model'].predict_proba(xs)[:,1]
        xval,yval,mval=fedkg_data.load_split('validation',policy=info['policy'],feature_set=info['feature_set'])
        pval=info['model'].predict_proba(xval)[:,1]
        thr=VAL_EVAL.best_f1_threshold(VAL_EVAL.pack(pval))
        met=test_e.metrics(test_e.pack(pv),None,float(thr))
        SECONDARY.append({'split':split,'model':key,'seed':42,'rows':len(ys),
                          'machines':ms.machineID.nunique(),'prevalence':float(ys.mean()),**met})
secondary_df=pd.DataFrame(SECONDARY)
secondary_df.to_csv(P3_DIR/'results'/'secondary_nested_time_window_checks.csv',index=False)
print(secondary_df.groupby(['split','model'])['ap'].mean().round(4).to_string())

In [ ]:
# Free temporary Phase 1/3 matrices and bootstrap packs before RDF graph construction.
import gc
for _name in ['T_X','T_Y','XtrA','XvaA','Xtr','Xv','Xtr_l','Xva_l','Xtr_b','Xva_b',
              'raw_feat','feat_all','SPLIT_DF','SPLIT_KEYS','CALIBRATION_KEYS','cand','tele','maint','errors',
              'SAVED_SCORE_PACKS','calibration_df','site_full_raw','SCALERS']:
    if _name in globals(): globals().pop(_name)
gc.collect()
print('Temporary in-memory matrices released; saved evidence and models remain on Drive.')

# Phase 4 — Five private-scope (logically separated) RDF Knowledge Graphs

Build one ontology and a separate RDF/Turtle graph per factory from the authoritative manifest. The graph contains only **observed records**: machines, error codes, maintenance components, failures and daily telemetry summaries. It does **not** assert unsupported causal relationships or guarantee cryptographic isolation. New as-of tests exclude events at the prediction timestamp as well as future events; daily aggregates are exposed only after their day has ended.

In [ ]:
import rdflib, re
OUT_DIR=PROJECT_DIR/'phase4_local_knowledge_graphs_v1'
for sub in ['ontology','graphs','tables','reports','logs']:(OUT_DIR/sub).mkdir(parents=True,exist_ok=True)
print('Source:',RAW_DIR,'KG outputs:',OUT_DIR)

In [ ]:
def locate_csv(*candidate_names):
    files = list(RAW_DIR.rglob("*.csv"))
    lookup = {p.name.casefold(): p for p in files}
    for name in candidate_names:
        if name.casefold() in lookup:
            return lookup[name.casefold()]
    wanted = {n.casefold() for n in candidate_names}
    found = [p for p in files if p.stem.casefold() in {Path(n).stem.casefold() for n in candidate_names}]
    if found:
        return found[0]
    raise FileNotFoundError(f"Could not find one of {candidate_names} under {RAW_DIR}. Available CSVs: {[p.name for p in files[:30]]}")

PATHS = {
    "telemetry": locate_csv("PdM_telemetry.csv", "telemetry.csv"),
    "errors": locate_csv("PdM_errors.csv", "errors.csv"),
    "maintenance": locate_csv("PdM_maint.csv", "PdM_maintenance.csv", "maintenance.csv", "maint.csv"),
    "failures": locate_csv("PdM_failures.csv", "failures.csv"),
    "machines": locate_csv("PdM_machines.csv", "machines.csv"),
}
print("Resolved inputs:")
for k, v in PATHS.items(): print(f"  {k:12s} {v}")

def normalize_id(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    try:
        return str(int(float(s)))
    except Exception:
        return s

def pick_column(df, candidates, required=True, table_name="table"):
    cols = {str(c).strip().casefold(): c for c in df.columns}
    for c in candidates:
        if c.casefold() in cols:
            return cols[c.casefold()]
    if required:
        raise ValueError(f"{table_name}: expected one of columns {candidates}; found {list(df.columns)}")
    return None

def load_source(name):
    df = pd.read_csv(PATHS[name], low_memory=False)
    machine_col = pick_column(df, ["machineID", "machine_id"], table_name=name)
    if machine_col != "machineID": df = df.rename(columns={machine_col: "machineID"})
    df["machineID"] = df["machineID"].map(normalize_id)
    dt_col = pick_column(df, ["datetime", "timestamp", "date"], required=(name != "machines"), table_name=name)
    if dt_col and dt_col != "datetime": df = df.rename(columns={dt_col: "datetime"})
    if "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
        if df["datetime"].isna().any():
            bad = int(df["datetime"].isna().sum())
            raise ValueError(f"{name}: {bad} invalid datetime values; resolve before KG creation.")
    return df

telemetry = load_source("telemetry")
errors = load_source("errors")
maintenance = load_source("maintenance")
failures = load_source("failures")
machines = load_source("machines")

SENSORS = ["volt", "rotate", "pressure", "vibration"]
missing_sensors = [c for c in SENSORS if c not in telemetry.columns]
if missing_sensors:
    raise ValueError(f"Telemetry is missing expected Azure sensor columns: {missing_sensors}; found {list(telemetry.columns)}")
for c in SENSORS:
    telemetry[c] = pd.to_numeric(telemetry[c], errors="coerce")
if telemetry[SENSORS].isna().any().any():
    raise ValueError("Sensor columns contain missing or non-numeric values; stop rather than silently inventing values.")

error_code_col = pick_column(errors, ["errorID", "error_id", "errorCode", "error_code"], table_name="errors")
if error_code_col != "errorID": errors = errors.rename(columns={error_code_col: "errorID"})
maintenance_comp_col = pick_column(maintenance, ["comp", "component", "componentID", "component_id"], table_name="maintenance")
if maintenance_comp_col != "comp": maintenance = maintenance.rename(columns={maintenance_comp_col: "comp"})
failure_comp_col = pick_column(failures, ["failure", "comp", "component", "componentID", "component_id", "failureType"], required=False, table_name="failures")
if failure_comp_col is not None and failure_comp_col != "failure_type":
    failures = failures.rename(columns={failure_comp_col: "failure_type"})
else:
    failures["failure_type"] = "unknown_component"

model_col = pick_column(machines, ["model", "machineModel", "machine_model"], required=False, table_name="machines")
age_col = pick_column(machines, ["age", "machineAge", "machine_age"], required=False, table_name="machines")
if model_col and model_col != "model": machines = machines.rename(columns={model_col: "model"})
if age_col and age_col != "age": machines = machines.rename(columns={age_col: "age"})
if "model" not in machines.columns: machines["model"] = "unknown_model"
if "age" not in machines.columns: machines["age"] = np.nan
machines["model"] = machines["model"].fillna("unknown_model").astype(str)
machines["age"] = pd.to_numeric(machines["age"], errors="coerce")

print({"telemetry_rows": len(telemetry), "error_rows": len(errors), "maintenance_rows": len(maintenance),
       "failure_rows": len(failures), "machine_rows": len(machines), "sensor_features": SENSORS})

In [ ]:
# Authoritative mapping read DIRECTLY from Phase 1 machine manifest.
manifest_disk=pd.read_csv(PREP_DIR/'manifests'/'factory_manifest.csv')
assert sha256_of(PREP_DIR/'manifests'/'factory_manifest.csv')==DATA_PROTOCOL['factory_manifest_sha256']
mapping_df=manifest_disk[['machineID','client_id']].copy()
mapping_df['machineID']=mapping_df['machineID'].map(normalize_id)
mapping_df['client_id']=mapping_df['client_id'].astype(str)
assert mapping_df['machineID'].is_unique
ids_per_machine=mapping_df.groupby('machineID')['client_id'].nunique()
factory_map=dict(zip(mapping_df['machineID'],mapping_df['client_id']))
EXPECTED_FACTORIES=[f'factory_{i}' for i in range(1,6)]
assert sorted(set(factory_map.values()))==EXPECTED_FACTORIES
assert set(machines.machineID)==set(factory_map)
assert mapping_df.groupby('client_id').size().eq(20).all()
mapping_df.to_csv(OUT_DIR/'tables'/'machine_factory_mapping.csv',index=False)
print('Partition inherited without re-computation:',mapping_df.groupby('client_id').size().to_dict())

In [ ]:
# Normalize event tables and retain the source's observed values only.
for df_name, df in [("errors", errors), ("maintenance", maintenance), ("failures", failures)]:
    if df["datetime"].isna().any():
        raise ValueError(f"{df_name}: invalid datetimes detected")
    unknown_machines = set(df["machineID"].dropna()) - set(factory_map)
    if unknown_machines:
        raise AssertionError(f"{df_name}: event rows refer to unmapped machines {sorted(unknown_machines)[:10]}")
    df["client_id"] = df["machineID"].map(factory_map)

errors["errorID"] = errors["errorID"].fillna("unknown_error").astype(str)
maintenance["comp"] = maintenance["comp"].fillna("unknown_component").astype(str)
failures["failure_type"] = failures["failure_type"].fillna("unknown_component").astype(str)
errors = errors.drop_duplicates(subset=["machineID", "datetime", "errorID"]).copy()
maintenance = maintenance.drop_duplicates(subset=["machineID", "datetime", "comp"]).copy()
failures = failures.drop_duplicates(subset=["machineID", "datetime", "failure_type"]).copy()

# One telemetry node per machine-day. It retains descriptive statistics, not every raw row.
telemetry["client_id"] = telemetry["machineID"].map(factory_map)
telemetry["day"] = telemetry["datetime"].dt.floor("D")
agg = {s: ["mean", "std", "min", "max"] for s in SENSORS}
daily = telemetry.groupby(["machineID", "client_id", "day"], as_index=False).agg(agg)
daily.columns = ["machineID", "client_id", "day"] + [f"{s}_{stat}" for s in SENSORS for stat in ["mean", "std", "min", "max"]]
daily = daily.merge(telemetry.groupby(["machineID", "client_id", "day"]).size().rename("n_samples").reset_index(),
                    on=["machineID", "client_id", "day"], how="left")

# Evidence tables are saved for later timestamp-filtered retrieval and audits.
for name, df in [("telemetry_daily_summaries", daily), ("error_events", errors),
                 ("maintenance_events", maintenance), ("failure_events", failures)]:
    to_save = df.copy()
    for c in to_save.columns:
        if pd.api.types.is_datetime64_any_dtype(to_save[c]):
            to_save[c] = to_save[c].map(lambda x: x.isoformat() if pd.notna(x) else "")
    to_save.to_csv(OUT_DIR / "tables" / f"{name}.csv", index=False)

print("Prepared factory-local evidence tables:")
print("daily telemetry summaries:", len(daily))
print("unique error occurrences:", len(errors))
print("unique maintenance events:", len(maintenance))
print("unique failure records:", len(failures))

In [ ]:
from rdflib import Graph, Namespace, RDF, RDFS, OWL, XSD, Literal, URIRef
from urllib.parse import quote

BASE = "https://example.org/fedkg-maint/"
ONT = Namespace(BASE + "ontology#")
RES = Namespace(BASE + "resource/")

CLASSES = {
    "Factory": "A factory partition used as a logical data boundary.",
    "Machine": "An industrial machine from the Azure predictive-maintenance dataset.",
    "MachineModel": "A model category recorded in the machine inventory.",
    "Component": "A component name observed in maintenance or failure records.",
    "ErrorCode": "An error code recorded by the source system.",
    "ErrorOccurrence": "An observed error event with its source timestamp and error code.",
    "MaintenanceEvent": "An observed maintenance record for a machine and component.",
    "FailureEvent": "An observed failure record from the source dataset.",
    "TelemetryDailySummary": "A per-machine, per-day aggregate of measured telemetry.",
}
OBJECT_PROPERTIES = {
    "hasMachine": ("Factory", "Machine", "Factory membership in this local graph."),
    "belongsToFactory": ("Machine", "Factory", "Factory assignment copied from Phase 1 metadata."),
    "usesModel": ("Machine", "MachineModel", "Recorded model category."),
    "hasErrorOccurrence": ("Machine", "ErrorOccurrence", "Observed error event; not a causal assertion."),
    "hasErrorCode": ("ErrorOccurrence", "ErrorCode", "Code recorded for an error occurrence."),
    "hasMaintenanceEvent": ("Machine", "MaintenanceEvent", "Observed maintenance event."),
    "targetsComponent": ("MaintenanceEvent", "Component", "Component named in the maintenance record."),
    "hasFailureEvent": ("Machine", "FailureEvent", "Observed failure event."),
    "failureOfComponent": ("FailureEvent", "Component", "Component label recorded for the failure."),
    "hasDailyTelemetry": ("Machine", "TelemetryDailySummary", "Daily telemetry summary for the machine."),
}
DATA_PROPERTIES = {
    "machineID": "Identifier from the source machine inventory.",
    "modelName": "Machine model category as recorded by the source.",
    "machineAge": "Machine age value from the source inventory.",
    "errorCode": "Error identifier from the source event record.",
    "componentName": "Component label from the source event record.",
    "occurredAt": "Source timestamp for an event.",
    "dailyDate": "Calendar date represented by a telemetry aggregate.",
    "nSamples": "Number of raw telemetry rows in the daily aggregate.",
}
for sensor in SENSORS:
    for stat in ["mean", "std", "min", "max"]:
        DATA_PROPERTIES[f"{sensor}{stat.title()}"] = f"Daily {stat} of the source telemetry field {sensor}."

ontology = Graph()
ontology.bind("fedkg", ONT); ontology.bind("rdf", RDF); ontology.bind("rdfs", RDFS); ontology.bind("owl", OWL); ontology.bind("xsd", XSD)
ontology.add((URIRef(BASE + "ontology"), RDF.type, OWL.Ontology))
ontology.add((URIRef(BASE + "ontology"), RDFS.label, Literal("FedKG-Maint Shared Maintenance Ontology")))
for name, description in CLASSES.items():
    term = ONT[name]
    ontology.add((term, RDF.type, OWL.Class)); ontology.add((term, RDFS.label, Literal(name)))
    ontology.add((term, RDFS.comment, Literal(description)))
for name, (domain, range_, description) in OBJECT_PROPERTIES.items():
    term = ONT[name]
    ontology.add((term, RDF.type, OWL.ObjectProperty)); ontology.add((term, RDFS.label, Literal(name)))
    ontology.add((term, RDFS.domain, ONT[domain])); ontology.add((term, RDFS.range, ONT[range_]))
    ontology.add((term, RDFS.comment, Literal(description)))
for name, description in DATA_PROPERTIES.items():
    term = ONT[name]
    ontology.add((term, RDF.type, OWL.DatatypeProperty)); ontology.add((term, RDFS.label, Literal(name)))
    ontology.add((term, RDFS.comment, Literal(description)))

ontology_path = OUT_DIR / "ontology" / "fedkg_shared_ontology.ttl"
ontology.serialize(destination=str(ontology_path), format="turtle")
ontology_json = {
    "ontology_iri": BASE + "ontology", "namespace": str(ONT),
    "classes": CLASSES,
    "object_properties": {k: {"domain": v[0], "range": v[1], "description": v[2]} for k, v in OBJECT_PROPERTIES.items()},
    "data_properties": DATA_PROPERTIES,
    "causal_assertions": "Not defined. Co-occurrence in these logs must not be described as proven causation.",
    "temporal_rule": "For an explanation at time T, retrieve event and telemetry facts with timestamps <= T only."
}
(OUT_DIR / "ontology" / "fedkg_shared_ontology.json").write_text(json.dumps(ontology_json, indent=2))
print("Shared ontology written:", ontology_path, "| triples:", len(ontology))

In [ ]:
def uri(factory_id, entity_type, key):
    return RES[f"{quote(str(factory_id), safe='')}/{quote(str(entity_type), safe='')}/{quote(str(key), safe='')}"]

def add_label(g, node, label):
    g.add((node, RDFS.label, Literal(str(label))))

def add_dt(g, subject, prop, value, datatype=None):
    if value is None or (isinstance(value, float) and not np.isfinite(value)) or pd.isna(value):
        return
    if datatype is not None:
        lit = Literal(value, datatype=datatype)
    else:
        lit = Literal(value)
    g.add((subject, ONT[prop], lit))

def event_datetime(value):
    ts = pd.Timestamp(value)
    return Literal(ts.isoformat(), datatype=XSD.dateTime)

# Inventory rows can have one row per machine; fail on conflicting duplicate records.
if machines["machineID"].duplicated().any():
    dupe = machines[machines["machineID"].duplicated(keep=False)]["machineID"].unique().tolist()
    raise AssertionError(f"Machine inventory has repeated machine IDs; review source before graph build: {dupe[:20]}")
machine_info = machines.set_index("machineID", drop=False).to_dict("index")

local_graphs = {}
graph_stats = []
for factory_id in EXPECTED_FACTORIES:
    g = Graph()
    g.bind("fedkg", ONT); g.bind("rdf", RDF); g.bind("rdfs", RDFS); g.bind("xsd", XSD)
    factory_node = uri(factory_id, "factory", factory_id)
    g.add((factory_node, RDF.type, ONT.Factory)); add_label(g, factory_node, factory_id)
    factory_machines = sorted([m for m, f in factory_map.items() if f == factory_id], key=lambda x: (not x.isdigit(), int(x) if x.isdigit() else x))

    for mid in factory_machines:
        mnode = uri(factory_id, "machine", mid)
        g.add((mnode, RDF.type, ONT.Machine)); add_label(g, mnode, f"Machine {mid}")
        g.add((factory_node, ONT.hasMachine, mnode)); g.add((mnode, ONT.belongsToFactory, factory_node))
        add_dt(g, mnode, "machineID", mid, XSD.string)
        info = machine_info.get(mid, {})
        model_name = str(info.get("model", "unknown_model"))
        model_node = uri(factory_id, "machine-model", model_name)
        g.add((model_node, RDF.type, ONT.MachineModel)); add_label(g, model_node, model_name)
        add_dt(g, model_node, "modelName", model_name, XSD.string)
        g.add((mnode, ONT.usesModel, model_node))
        add_dt(g, mnode, "modelName", model_name, XSD.string)
        age = info.get("age", np.nan)
        if pd.notna(age): add_dt(g, mnode, "machineAge", float(age), XSD.float)

    sub = daily[daily["client_id"] == factory_id]
    for _, row in sub.iterrows():
        mid = row["machineID"]
        mnode = uri(factory_id, "machine", mid)
        d = pd.Timestamp(row["day"])
        dstr = d.strftime("%Y-%m-%d")
        node = uri(factory_id, "telemetry-daily", f"{mid}/{dstr}")
        g.add((node, RDF.type, ONT.TelemetryDailySummary)); add_label(g, node, f"Telemetry daily summary for machine {mid} on {dstr}")
        g.add((mnode, ONT.hasDailyTelemetry, node))
        add_dt(g, node, "dailyDate", dstr, XSD.date)
        add_dt(g, node, "nSamples", int(row["n_samples"]), XSD.integer)
        for sensor in SENSORS:
            for stat in ["mean", "std", "min", "max"]:
                prop = f"{sensor}{stat.title()}"
                add_dt(g, node, prop, float(row[f"{sensor}_{stat}"]), XSD.float)

    local_errors = errors[errors["client_id"] == factory_id]
    for _, row in local_errors.iterrows():
        mid, code, ts = row["machineID"], str(row["errorID"]), pd.Timestamp(row["datetime"])
        event_key = f"{mid}/{ts.isoformat()}/{code}"
        ev = uri(factory_id, "error-occurrence", event_key)
        mnode = uri(factory_id, "machine", mid)
        code_node = uri(factory_id, "error-code", code)
        g.add((ev, RDF.type, ONT.ErrorOccurrence)); add_label(g, ev, f"Error {code} on machine {mid} at {ts.isoformat()}")
        g.add((mnode, ONT.hasErrorOccurrence, ev)); g.add((ev, ONT.hasErrorCode, code_node))
        g.add((code_node, RDF.type, ONT.ErrorCode)); add_label(g, code_node, code)
        add_dt(g, code_node, "errorCode", code, XSD.string)
        g.add((ev, ONT.occurredAt, event_datetime(ts)))

    local_maint = maintenance[maintenance["client_id"] == factory_id]
    for _, row in local_maint.iterrows():
        mid, comp, ts = row["machineID"], str(row["comp"]), pd.Timestamp(row["datetime"])
        ev = uri(factory_id, "maintenance-event", f"{mid}/{ts.isoformat()}/{comp}")
        mnode = uri(factory_id, "machine", mid)
        cnode = uri(factory_id, "component", comp)
        g.add((ev, RDF.type, ONT.MaintenanceEvent)); add_label(g, ev, f"Maintenance on machine {mid}, component {comp}, at {ts.isoformat()}")
        g.add((mnode, ONT.hasMaintenanceEvent, ev)); g.add((ev, ONT.targetsComponent, cnode))
        g.add((cnode, RDF.type, ONT.Component)); add_label(g, cnode, comp); add_dt(g, cnode, "componentName", comp, XSD.string)
        g.add((ev, ONT.occurredAt, event_datetime(ts)))

    local_fail = failures[failures["client_id"] == factory_id]
    for _, row in local_fail.iterrows():
        mid, comp, ts = row["machineID"], str(row["failure_type"]), pd.Timestamp(row["datetime"])
        ev = uri(factory_id, "failure-event", f"{mid}/{ts.isoformat()}/{comp}")
        mnode = uri(factory_id, "machine", mid)
        cnode = uri(factory_id, "component", comp)
        g.add((ev, RDF.type, ONT.FailureEvent)); add_label(g, ev, f"Observed failure on machine {mid}, label {comp}, at {ts.isoformat()}")
        g.add((mnode, ONT.hasFailureEvent, ev)); g.add((ev, ONT.failureOfComponent, cnode))
        g.add((cnode, RDF.type, ONT.Component)); add_label(g, cnode, comp); add_dt(g, cnode, "componentName", comp, XSD.string)
        g.add((ev, ONT.occurredAt, event_datetime(ts)))

    out_path = OUT_DIR / "graphs" / f"{factory_id}.ttl"
    g.serialize(destination=str(out_path), format="turtle")
    local_graphs[factory_id] = g
    graph_stats.append({
        "factory": factory_id,
        "machines": len(factory_machines),
        "daily_telemetry_nodes": len(sub),
        "error_occurrences": len(local_errors),
        "maintenance_events": len(local_maint),
        "failure_records": len(local_fail),
        "rdf_triples": len(g),
        "ttl_size_mb": round(out_path.stat().st_size / (1024**2), 3),
        "graph_path": str(out_path),
    })

stats_df = pd.DataFrame(graph_stats)
stats_df.to_csv(OUT_DIR / "reports" / "factory_graph_statistics.csv", index=False)
print(stats_df.to_string(index=False))

In [ ]:
# Validation checks: structural and partition checks, not model-performance claims.
checks = []
def check(name, ok, detail=""):
    checks.append({"check": name, "status": "PASS" if bool(ok) else "FAIL", "detail": str(detail)})
    print(f"[{ 'PASS' if ok else 'FAIL' }] {name}: {detail}")

check("V1 source dataset is Azure tables only", all("PdM_" in str(p) or p.name.lower() in {"telemetry.csv", "errors.csv", "maintenance.csv", "maint.csv", "failures.csv", "machines.csv"} for p in PATHS.values()),
      "No NASA C-MAPSS input is loaded in this notebook.")
check("V2 five expected factory partitions are present", set(local_graphs) == set(EXPECTED_FACTORIES), str(sorted(local_graphs)))
check("V3 each source machine maps to exactly one factory", bool((ids_per_machine == 1).all()) and mapping_df["machineID"].nunique() == len(mapping_df), f"mapped machines={mapping_df['machineID'].nunique()}")
check("V4 machine coverage matches the source inventory", set(machines["machineID"]) <= set(factory_map), f"source inventory machines={machines['machineID'].nunique()}")
check("V5 shared ontology file exists", ontology_path.exists() and len(ontology) > 0, f"ontology triples={len(ontology)}")
check("V6 all local graph files exist", all((OUT_DIR / "graphs" / f"{f}.ttl").exists() for f in EXPECTED_FACTORIES), "one RDF/Turtle file per factory")

for f in EXPECTED_FACTORIES:
    g = local_graphs[f]
    expected = {m for m, assigned in factory_map.items() if assigned == f}
    actual = {str(g.value(node, ONT.machineID)) for node in g.subjects(RDF.type, ONT.Machine) if g.value(node, ONT.machineID) is not None}
    check(f"V7 {f} machine isolation", actual == expected, f"expected={len(expected)} actual={len(actual)}")
    error_nodes = set(g.subjects(RDF.type, ONT.ErrorOccurrence))
    maint_nodes = set(g.subjects(RDF.type, ONT.MaintenanceEvent))
    failure_nodes = set(g.subjects(RDF.type, ONT.FailureEvent))
    bad_err = [ev for ev in error_nodes if not list(g.subjects(ONT.hasErrorOccurrence, ev)) or not list(g.objects(ev, ONT.hasErrorCode)) or not list(g.objects(ev, ONT.occurredAt))]
    bad_maint = [ev for ev in maint_nodes if not list(g.subjects(ONT.hasMaintenanceEvent, ev)) or not list(g.objects(ev, ONT.targetsComponent)) or not list(g.objects(ev, ONT.occurredAt))]
    bad_fail = [ev for ev in failure_nodes if not list(g.subjects(ONT.hasFailureEvent, ev)) or not list(g.objects(ev, ONT.failureOfComponent)) or not list(g.objects(ev, ONT.occurredAt))]
    check(f"V8 {f} error events linked", not bad_err, f"events={len(error_nodes)} broken={len(bad_err)}")
    check(f"V9 {f} maintenance events linked", not bad_maint, f"events={len(maint_nodes)} broken={len(bad_maint)}")
    check(f"V10 {f} failure records linked", not bad_fail, f"events={len(failure_nodes)} broken={len(bad_fail)}")
    tele_nodes = set(g.subjects(RDF.type, ONT.TelemetryDailySummary))
    bad_tele = [n for n in tele_nodes if not list(g.subjects(ONT.hasDailyTelemetry, n)) or not list(g.objects(n, ONT.dailyDate))]
    check(f"V11 {f} daily telemetry linked", not bad_tele, f"summaries={len(tele_nodes)} broken={len(bad_tele)}")
    # No causal relation names are declared in the shared ontology.
    causal_props = [p for p in ontology.subjects(RDF.type, OWL.ObjectProperty) if any(t in str(p).lower() for t in ["causes", "causedby", "leadsTo".lower()])]
    check(f"V12 {f} ontology avoids unsupported causal edges", len(causal_props) == 0, "relationships express observations/records only")

checks_df = pd.DataFrame(checks)
checks_df.to_csv(OUT_DIR / "reports" / "phase4_kg_validation_checks.csv", index=False)
failed = checks_df[checks_df["status"] != "PASS"]
if len(failed):
    display(failed)
    raise AssertionError(f"Phase 4 has {len(failed)} failed validation checks; do not treat the graph as closed.")
print(f"\nAll {len(checks_df)} structural checks passed.")

In [ ]:
def _rdf_timestamp(v):
    if v is None: return None
    try:return pd.Timestamp(str(v))
    except Exception:return None

def retrieve_as_of(factory_id,machine_id,as_of,lookback_days=30):
    """Return strictly prior observed events and only completed-day telemetry summaries.
    This is as-of fact retrieval, NOT a causal or LLM recommendation engine.
    """
    if factory_id not in local_graphs:raise KeyError(factory_id)
    mid=normalize_id(machine_id)
    if factory_map.get(mid)!=factory_id:raise PermissionError('Cross-factory graph access rejected')
    g=local_graphs[factory_id]
    as_of=pd.Timestamp(as_of);lower=as_of-pd.Timedelta(days=int(lookback_days))
    mnode=uri(factory_id,'machine',mid)
    assert (mnode,RDF.type,ONT.Machine) in g
    out=[]
    for prop,kind,label_prop in [(ONT.hasErrorOccurrence,'error',ONT.hasErrorCode),
                                  (ONT.hasMaintenanceEvent,'maintenance',ONT.targetsComponent),
                                  (ONT.hasFailureEvent,'failure',ONT.failureOfComponent)]:
        for ev in g.objects(mnode,prop):
            t=_rdf_timestamp(g.value(ev,ONT.occurredAt))
            if t is None or not (lower<=t<as_of):continue
            ref=g.value(ev,label_prop)
            descriptor=str(g.value(ref,ONT.errorCode) or g.value(ref,ONT.componentName) or g.value(ref,RDFS.label) or 'unknown')
            out.append({'kind':kind,'timestamp':t.isoformat(),'observed_value':descriptor})
    for node in g.objects(mnode,ONT.hasDailyTelemetry):
        d=_rdf_timestamp(g.value(node,ONT.dailyDate))
        # A full-day mean is not observable until the day finishes, so exclude current day entirely.
        if d is None or not (lower.normalize()<=d.normalize()<as_of.normalize()):continue
        out.append({'kind':'daily_telemetry','timestamp':d.isoformat(),
                    'volt_mean':float(g.value(node,ONT.voltMean)),
                    'rotate_mean':float(g.value(node,ONT.rotateMean)),
                    'pressure_mean':float(g.value(node,ONT.pressureMean)),
                    'vibration_mean':float(g.value(node,ONT.vibrationMean))})
    return sorted(out,key=lambda v:v['timestamp'])

# Adversarial as-of checks: sampling across factory boundaries and failure timestamps.
from collections import Counter
KG_CHECKS=[]
def kg_assert(name,ok,detail=''):
    KG_CHECKS.append({'check':name,'status':'PASS' if ok else 'FAIL','detail':str(detail)})
    if not ok:raise AssertionError(name+': '+str(detail))
    print('[PASS]',name,detail)
kg_assert('KG-A Manifest unchanged',sha256_of(PREP_DIR/'manifests'/'factory_manifest.csv')==DATA_PROTOCOL['factory_manifest_sha256'])
all_graph_machines={f:set(mapping_df.loc[mapping_df.client_id==f,'machineID']) for f in EXPECTED_FACTORIES}
for f in EXPECTED_FACTORIES:
    g=local_graphs[f]
    ms={str(g.value(node,ONT.machineID)) for node in g.subjects(RDF.type,ONT.Machine)}
    kg_assert(f'KG-B {f} isolation',ms==all_graph_machines[f],f'{len(ms)} machines')
# Cross-factory access must be rejected.
try:
    retrieve_as_of('factory_1',next(iter(all_graph_machines['factory_5'])),'2015-12-01')
    rejected=False
except PermissionError:rejected=True
kg_assert('KG-C cross-factory access rejected',rejected)
probe=failures[['machineID','datetime']].drop_duplicates().sort_values('datetime').iloc[::max(1,len(failures)//100)]
assert len(probe)>0
for _,r in probe.iterrows():
    m=normalize_id(r.machineID);f=factory_map[m];t=pd.Timestamp(r.datetime)
    result=retrieve_as_of(f,m,t,lookback_days=365)
    for fact in result:
        ts=pd.Timestamp(fact['timestamp'])
        if fact['kind']!='daily_telemetry':
            assert ts<t, f'Current/future event leaked into retrieval for {m}, {t}'
        else:
            assert ts.normalize()<t.normalize(),f'Partial current day leaked for {m}, {t}'
kg_assert('KG-D adversarial failure-time no-peek audit',True,f'{len(probe)} randomly spaced failure-time probes, strict earlier-than as_of')
kg_summary={'graphs':graph_stats,'structural_checks':checks_df.to_dict('records'),'asof_checks':KG_CHECKS,
            'limitations':['No causality inferred','Graph not used to train predictive classifier',
                'Raw source data are centrally accessible in this Colab simulation',
                'Only completed-day aggregates are retrievable']}
(OUT_DIR/'reports'/'integrated_kg_audit.json').write_text(json.dumps(kg_summary,indent=2,default=str))
print('All local graphs and as-of audits completed.')

# Final integrated audit and decision

A pass is granted only for implemented stages and validated conditions. Checklists distinguish missing future work (LLM explanation, real distributed deployment, NASA-specific experiment, independent sites) from faults in the Phase-1-through-4 notebook. Numerical results are generated exclusively from this run's saved predictions and held-out labels. A `RUN_COMPLETE.json` marker is created only after the pipeline reaches this point.

In [ ]:
# Integrity audit over the single coherent run folder; do not merge previous output versions.
validation_checks = {
    'P1_raw_hashes_locked':(MAN_DIR/'raw_data_checksums.json').exists(),
    'P1_split_artifact_hashes_unchanged':all(sha256_of(PROC_DIR/name)==h for name,h in DATA_PROTOCOL['split_artifact_sha256'].items()),
    'P1_no_future_label_leakage':all(v['status']=='PASS' for v in CHECKS.values()),
    'P1_25_full_features':len(ALL_FEATURES)==25,
    'P1_calibration_without_targets':TARGET not in pd.read_parquet(PROC_DIR/'heldout_calibration_features.parquet').columns,
    'P2_model_selection_lock':(P2_DIR/'reports'/'validation_only_selection_lock.json').exists(),
    'P3_final_selection_lock':LOCK_PATH.exists(),
    'P3_local_norm_policy':MODEL_POLICY=='local',
    'P3_two_distinct_FL_strategies':all(m in FL_ARTIFACTS for m in [('FedAvg',42),('FedProx',42)]),
    'P3_DP_accounted_within_cap':max(EPSILON_MANUAL,EPSILON_LIBRARY)<=EPSILON_CAP,
    'P3_results_per_seed':len(metrics_df[metrics_df.model=='DP-FedAvg'])==len(SEEDS),
    'P3_paired_bootstrap_complete':len(paired)==3,
    'P3_2_3_4_factory_ablation':set(client_count_df.clients)=={2,3,4},
    'P4_ontology_exported':ontology_path.exists(),
    'P4_five_graphs_exported':all((OUT_DIR/'graphs'/f'{f}.ttl').exists() for f in EXPECTED_FACTORIES),
    'P4_graph_checks_passed':all(c['status']=='PASS' for c in checks_df.to_dict('records')+KG_CHECKS)
}
assert all(validation_checks.values()),f'Invalid pipeline checks: {[k for k,v in validation_checks.items() if not v]}'
model_summary=metrics_df.groupby('model',as_index=False).agg(mean_pr_auc=('ap','mean'),
                 sd_pr_auc=('ap','std'),mean_roc_auc=('roc_auc','mean'),seeds=('seed','nunique')).sort_values('mean_pr_auc',ascending=False)
model_summary.to_csv(PROJECT_DIR/'final_model_summary.csv',index=False)
limitations=[
    'Azure source only; NASA C-MAPSS was not evaluated.',
    'The five factories are simulated, not independently collected industrial sites.',
    'Held-out factory normalization uses unlabeled historical calibration; not zero-shot.',
    'Previous project iterations used held-out factory; no claim of untouched external confirmation.',
    'FP32 payload count excludes network, compression, authentication and secure aggregation overhead.',
    'Only client-update DP release is accounted; graphs, preprocessing and evaluator are not private.',
    'Local-only ensemble is a reference, not a second FL strategy.',
    'No real secure aggregation or graph-based predictive training is claimed.',
    'Knowledge Graph does not establish causality; LLM explanations are beyond Phase 4.',
    'Three seeds and 20 site machines do not establish universal factory generalization.',
    'Alternative type-blocked manifest was saved but not retrained in this run.'
]
final_report={'run_id':RUN_ID,'generated_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'complete':True,'mode':'smoke' if FAST_SMOKE else 'full','checks':validation_checks,
    'source_files':'Microsoft Azure PdM telemetry, errors, failures, maint, machines',
    'data_protocol_sha256':sha256_of(PROJECT_DIR/'data_protocol_lock.json'),
    'selection_lock_sha256':sha256_of(LOCK_PATH),
    'primary_eval':'factory_5 after unlabeled calibration; test_site_full',
    'models':model_summary.to_dict('records'),'machine_cluster_paired_ci':paired,
    'dp_mechanism':PRIVACY_SCOPE,'communication':COMM,'client_count_sensitivity':client_count_df.to_dict('records'),'kg':kg_summary,
    'limitations':limitations,'not_implemented':['NASA C-MAPSS','LLM grounded explanations','real distributed confidential deployment','external independent factory evaluation']}
(PROJECT_DIR/'FINAL_INTEGRATED_AUDIT.json').write_text(json.dumps(final_report,indent=2,default=str))
lines=['# FedKG-Maint — Unified Run Results','',f'Run: `{RUN_ID}`',f'Mode: {final_report["mode"]}',
       '','## Compliance checks','']
lines += [f'- {k}: {"PASS" if v else "FAIL"}' for k,v in validation_checks.items()]
lines += ['','## Held-out primary results (PR-AUC)',
          '', '| Model | Mean PR-AUC | SD across available seeds | Number of seeds |', '|---|---:|---:|---:|']
for r in model_summary.to_dict('records'):
    lines.append(f'| {r["model"]} | {r["mean_pr_auc"]:.5f} | {r["sd_pr_auc"]:.5f} | {r["seeds"]} |')
lines += ['','## Mechanism-only DP',f'- ε conservative: {EPSILON_MANUAL:.6f}; δ: {DP_DELTA}',
          f'- Independent accountant ε: {EPSILON_LIBRARY:.6f}',
          '- No end-to-end DP guarantee.','', '## Boundaries']+[f'- {x}' for x in limitations]
(PROJECT_DIR/'FINAL_INTEGRATED_REPORT.md').write_text('\n'.join(lines))
(PROJECT_DIR/'RUN_COMPLETE.json').write_text(json.dumps({'run_id':RUN_ID,'time_utc':final_report['generated_utc'],
    'sha256':sha256_of(PROJECT_DIR/'FINAL_INTEGRATED_AUDIT.json')},indent=2))
print(model_summary.round(5).to_string(index=False))
print('Final reports:',PROJECT_DIR/'FINAL_INTEGRATED_REPORT.md',PROJECT_DIR/'FINAL_INTEGRATED_AUDIT.json')
print('RUN COMPLETE:',RUN_ID,'| Checks',sum(validation_checks.values()),'/',len(validation_checks))

## Scope boundary / experiment record

**The full run requires the five raw Azure CSVs on Drive (or downloads them into the Drive raw folder), Python dependencies, and enough memory to hold the model matrices and RDF graphs.** Saved outputs are evidence only after their cells execute. `FAST_SMOKE` is only for execution checks, never for scientific findings. Previous Phase 3 mean PR-AUC values and previous PASS reports are deliberately not pre-populated. When publishing results, provide the `FINAL_SELECTION_LOCK.json`, frozen raw hashes, processed split hashes, saved predictions, seeds and the complete run report.